# Sit.3 — Entrenar Conv3D (CNN 3D) + Covariables ERA5/MODIS

**Flujo completo:**
1. Descarga dataset + paneles ERA5/MODIS desde HuggingFace
2. Extrae features meteorologicas (T2m, BLH, viento, presion, precipitacion) + MODIS AOD
3. Inyecta 9 canales extra al tensor → (1403, 8, 531, 5, 5)
4. Entrena Conv3DSit3 con 531 canales
5. Evalua y compara vs baseline de 522 canales
6. **LOO-CV con DAGMA** (validacion honesta por estacion) + ensemble persistencia/EWMA

Dataset: `Slucu-0310/geovision-cali-sit3` + `Slucu-0310/geovision-cali-panel`

---

## Nota metodologica importante (leer antes de interpretar metricas)

El split 70/15/15 aleatorio original (`train_test_split(np.arange(N), ...)`)
tiene **data leakage** porque:

- Las 1403 secuencias se construyen con ventana deslizante de 8 timesteps
  sobre tiles que **comparten ubicacion**; secuencias consecutivas comparten
  7/8 inputs y targets casi identicos por autocorrelacion.
- Al repartir aleatoriamente, train y test contienen vecinos casi clonados →
  el modelo memoriza el tile y "predice" valores casi perfectos
  (ej. NO2 RMSE=0.08 µg/m³, fisicamente imposible: el ruido propio
  de DAGMA ya es ~3 µg/m³).

Por eso esta version corregida del notebook hace:

1. **Split honesto**: por bloques contiguos del tensor (proxy de tile_id),
   sin solapamiento de ventanas entre splits. Los numeros "test" bajaran
   pero seran reales.
2. **Validacion principal = LOO-CV por estacion DAGMA** (celdas 24-27).
   Esa es la que se compara con KPI. Usa persistencia + correccion ligera
   del Conv3D, con clipping a rango plausible.
3. **Curvas de entrenamiento** consolidadas en una sola imagen
   (`training_curves.png`).

> El target `y` viene de Sentinel-5P (columnas troposfericas) y DAGMA mide
> superficie. Son **cantidades correlacionadas pero distintas**; parte del
> RMSE LOO-CV es ese mismatch fisico, no del modelo. Documentar en informe.


In [1]:
# @title DIAGNOSTICO: ver que hay en /kaggle/input/
import os
from pathlib import Path

kaggle = Path("/kaggle/input")
print(f"Existe /kaggle/input? {kaggle.exists()}")
if kaggle.exists():
    print(f"\nDirectorios en /kaggle/input/:")
    for d in sorted(kaggle.iterdir()):
        if d.is_dir():
            print(f"\n  {d.name}/")
            # Mostrar primeros 20 archivos
            all_files = list(d.rglob("*"))
            for f in all_files[:20]:
                if f.is_file():
                    size = f.stat().st_size / 1024
                    print(f"    {f.relative_to(d)} ({size:.1f} KB)")
            if len(all_files) > 20:
                print(f"    ... y {len(all_files) - 20} archivos mas")
else:
    print("/kaggle/input/ no existe! Estas en Kaggle?")

print(f"\nBusqueda especifica:")
for pattern in ["metadatos.parquet", "DAGMA*", "grid_embed*", "*.parquet", "*.npz"]:
    found = list(Path("/").rglob(pattern))[:5]
    if found:
        print(f"  {pattern}: {[str(f) for f in found]}")
    else:
        print(f"  {pattern}: no encontrado")


Existe /kaggle/input? True

Directorios en /kaggle/input/:

  datasets/
    samuelpatino/dagma-data/DAGMA_con_Acopi_NO2.parquet (2404.9 KB)
    samuelpatino/geovision-cali-sit2/metadatos.parquet (326.9 KB)
    samuelpatino/geovision-cali-sit2/grid_embeddings_sit2.npz (105131.6 KB)

Busqueda especifica:
  metadatos.parquet: ['/kaggle/input/datasets/samuelpatino/geovision-cali-sit2/metadatos.parquet']
  DAGMA*: ['/kaggle/input/datasets/samuelpatino/dagma-data/DAGMA_con_Acopi_NO2.parquet']
  grid_embed*: ['/kaggle/input/datasets/samuelpatino/geovision-cali-sit2/grid_embeddings_sit2.npz']
  *.parquet: ['/usr/local/lib/python3.12/dist-packages/pyarrow/tests/data/parquet/v0.7.1.parquet', '/usr/local/lib/python3.12/dist-packages/pyarrow/tests/data/parquet/v0.7.1.some-named-index.parquet', '/usr/local/lib/python3.12/dist-packages/pyarrow/tests/data/parquet/v0.7.1.all-named-index.parquet', '/usr/local/lib/python3.12/dist-packages/pyarrow/tests/data/parquet/v0.7.1.column-metadata-handling.parque

In [2]:
# @title Dependencias
%pip install -q huggingface_hub numpy pandas tqdm matplotlib
%pip install -q torch lightning

import warnings, os, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")

try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
pl.seed_everything(SEED, workers=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")



Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.4/848.4 kB 23.6 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


Seed set to 42


Device: cuda


Device: cuda


In [3]:
# @title Pipeline Covariables (ERA5 + MODIS AOD)
%pip install -q zarr xarray scipy
# Descarga paneles, extrae features, e inyecta 9 canales extra
# Genera tensor con 531 canales en dataset_sit3_extra/

import os, sys, warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
from scipy.interpolate import RegularGridInterpolator
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from huggingface_hub import snapshot_download

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# ---- Configurar rutas ----
# Tus datasets de Kaggle: geovision-cali-sit2 y dagma-data
# Asegurate de agregarlos como INPUTS en el notebook (panel derecho -> Add Data)

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_SIT2 = KAGGLE_INPUT / "geovision-cali-sit2"

# DIAGNOSTICO: listar todo en /kaggle/input/
if KAGGLE_INPUT.exists():
    print("  === Contenido de /kaggle/input/ ===")
    for d in sorted(KAGGLE_INPUT.iterdir()):
        if d.is_dir():
            print(f"  Dir: {d.name}/")
            for f in sorted(d.rglob("*")):
                if f.is_file():
                    kb = f.stat().st_size / 1024
                    print(f"    {f.relative_to(d)} ({kb:.0f} KB)")
else:
    print("  ERROR: /kaggle/input/ NO EXISTE")

# Prioridad 1: ruta exacta de Kaggle
if (KAGGLE_SIT2 / "metadatos.parquet").exists():
    DATA_DIR = KAGGLE_SIT2
    print(f"  DATA_DIR = {DATA_DIR} (Kaggle)")
else:
    # Prioridad 2: buscar con rglob
    found = list(KAGGLE_INPUT.rglob("metadatos.parquet"))
    if found:
        DATA_DIR = found[0].parent
        print(f"  DATA_DIR = {DATA_DIR} (Kaggle rglob)")
    else:
        DATA_DIR = Path("/content/dataset_sit2")
        print(f"  DATA_DIR = {DATA_DIR} (fallback)")

# SIT3_DIR: buscar tensor en Kaggle o usar HF descarga
if (KAGGLE_SIT2 / "X_convlstm.npy").exists():
    SIT3_DIR = KAGGLE_SIT2
    print(f"  SIT3_DIR = {SIT3_DIR} (Kaggle)")
else:
    found = list(KAGGLE_INPUT.rglob("X_convlstm.npy"))
    if found:
        SIT3_DIR = found[0].parent
        print(f"  SIT3_DIR = {SIT3_DIR} (Kaggle rglob)")
    else:
        SIT3_DIR = Path("/content/dataset_sit3")
        print(f"  SIT3_DIR = {SIT3_DIR} (se descargara de HF)")

PANEL_DIR = Path("/content/panels")
OUT_DIR = Path("/content/dataset_sit3_extra")

print(f"\n  Verificando rutas:")
print(f"    metadatos.parquet en DATA_DIR: {(DATA_DIR / 'metadatos.parquet').exists()}")
print(f"    X_convlstm.npy en SIT3_DIR:   {(SIT3_DIR / 'X_convlstm.npy').exists()}")

CELL_RES = 0.05
SEQ_LEN = 8
N_COV = 9

COV_CHANNELS = [
    "era5_t2m", "era5_blh", "era5_wind_speed",
    "era5_sp", "era5_tp", "modis_aod_055",
    "era5_t2m_delta", "era5_blh_delta", "era5_wind_speed_delta",
]

# ============================================================
# 0. Descargar paneles desde HF (si no estan en Kaggle)
# ============================================================
print("=" * 60)
print("  [0] Descargando paneles ERA5 y MODIS...")
print("=" * 60)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

for folder in ["ERA-5", "MODIS_MCD"]:
    panel_path = PANEL_DIR / folder / "panel.zarr"
    if panel_path.is_dir():
        print(f"  {folder}: ya existe en {panel_path}")
        continue
    # Si PANEL_DIR es un Kaggle dataset, intentar ahi
    if PANEL_DIR != Path("/content/panels") and (PANEL_DIR / folder / "panel.zarr").is_dir():
        print(f"  {folder}: encontrado en Kaggle")
        continue
    # Descargar de HF
    print(f"  {folder}: descargando desde HuggingFace...")
    PANEL_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id="Slucu-0310/geovision-cali-panel",
        repo_type="dataset",
        allow_patterns=f"{folder}/panel.zarr/*",
        local_dir=str(PANEL_DIR),
        token=HF_TOKEN,
    )
print("  OK")

# ============================================================
# 0b. Descargar tensor desde HF (si no esta en Kaggle)
# ============================================================
if not (SIT3_DIR / "X_convlstm.npy").exists():
    print("\n  Tensor no encontrado en Kaggle, descargando desde HuggingFace...")
    SIT3_DIR = Path("/content/dataset_sit3")
    SIT3_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id="Slucu-0310/geovision-cali-sit3",
        repo_type="dataset",
        local_dir=str(SIT3_DIR),
        token=HF_TOKEN,
    )
    print(f"  Tensor descargado a {SIT3_DIR}")

# ============================================================
# 1. Cargar metadatos y tensor existente
# ============================================================
print()
print("  [1] Cargando metadatos...")
df = pd.read_parquet(DATA_DIR / "metadatos.parquet")
df["fecha_dt"] = pd.to_datetime(df["fecha"])
print(f"  Metadatos: {len(df)} tiles")

X_existing = np.load(SIT3_DIR / "X_convlstm.npy", mmap_mode="r")
print(f"  X existente: {X_existing.shape}")

# ============================================================
# 2. Extraer ERA5 features
# ============================================================
print()
print("  [2] Extrayendo ERA5...")
era5 = xr.open_zarr(str(PANEL_DIR / "ERA-5" / "panel.zarr"), consolidated=True)
era5_times = pd.to_datetime(era5.time.values)
era5_y = era5.y.values
era5_x = era5.x.values
era5_data = era5["data"].values

bands = list(era5.band.values)
i_t2m = bands.index("temperature_2m")
i_blh = bands.index("boundary_layer_height")
i_sp = bands.index("surface_pressure")
i_tp = bands.index("total_precipitation")
i_u = bands.index("u_component_of_wind_10m")
i_v = bands.index("v_component_of_wind_10m")

era5_rows = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="ERA5"):
    lat, lon = row.centroide_lat, row.centroide_lon
    yi = int(np.argmin(np.abs(era5_y - lat)))
    xi = int(np.argmin(np.abs(era5_x - lon)))
    ti = int(np.argmin(np.abs(era5_times.values - np.datetime64(row.fecha_dt))))

    t2m = float(era5_data[ti, i_t2m, yi, xi])
    blh = float(era5_data[ti, i_blh, yi, xi])
    sp  = float(era5_data[ti, i_sp, yi, xi])
    tp  = float(era5_data[ti, i_tp, yi, xi])
    u10 = float(era5_data[ti, i_u, yi, xi])
    v10 = float(era5_data[ti, i_v, yi, xi])
    ws  = np.sqrt(u10**2 + v10**2)

    era5_rows.append({
        "era5_t2m": (t2m - 292.0) / 5.0 if np.isfinite(t2m) else 0,
        "era5_blh": blh / 1000.0 if np.isfinite(blh) else 0,
        "era5_wind_speed": ws / 5.0 if np.isfinite(ws) else 0,
        "era5_sp": (sp - 88000) / 2000.0 if np.isfinite(sp) else 0,
        "era5_tp": tp * 1000.0 if np.isfinite(tp) else 0,
    })

era5_df = pd.DataFrame(era5_rows)
# Deltas y rolling
for col in list(era5_df.columns):
    era5_df[f"{col}_delta"] = era5_df[col].diff().fillna(0)
for col in list(era5_df.columns):
    if not col.endswith("_delta"):
        era5_df[f"{col}_roll3"] = era5_df[col].rolling(3, min_periods=1).mean()

for col in era5_df.columns:
    df[col] = era5_df[col].values
print(f"  OK: {len([c for c in df.columns if c.startswith('era5')])} columnas ERA5")

# ============================================================
# 3. Extraer MODIS AOD
# ============================================================
print()
print("  [3] Extrayendo MODIS AOD...")
modis = xr.open_zarr(str(PANEL_DIR / "MODIS_MCD" / "panel.zarr"), consolidated=True)

# Parsear fechas de nombres MODIS
raw_t = modis.time.values
m_dates = []
for t in raw_t:
    parts = str(t).split("_")
    if len(parts) >= 2:
        try:
            d = parts[1][1:9]
            m_dates.append(np.datetime64(f"{d[:4]}-{d[4:7]}"))
        except:
            m_dates.append(np.datetime64("NaT"))
    else:
        m_dates.append(np.datetime64("NaT"))
m_dates = np.array(m_dates)
valid_mask = ~np.isnat(m_dates)

aod_idx = list(modis.band.values).index("Optical_Depth_055")
modis_y = modis.y.values
modis_x = modis.x.values

aod_vals = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="MODIS"):
    lat_c, lon_c = row.centroide_lat, row.centroide_lon
    target = np.datetime64(row.fecha_dt)
    diff = np.abs(m_dates - target)
    diff[~valid_mask] = np.timedelta64(999, "D")
    nearest = int(np.argmin(diff))

    if diff[nearest] > np.timedelta64(5, "D"):
        aod_vals.append(0.0)
        continue

    aod_slice = modis["data"].isel(time=nearest, band=aod_idx).values
    aod_clean = np.nan_to_num(aod_slice, nan=0.0)

    if not np.isfinite(aod_clean).any():
        aod_vals.append(0.0)
        continue

    # Interpolar a grilla 5x5 del tile
    lats_tile = np.linspace(lat_c - 2*0.005, lat_c + 2*0.005, 5)
    lons_tile = np.linspace(lon_c - 2*0.005, lon_c + 2*0.005, 5)
    try:
        interp = RegularGridInterpolator(
            (modis_y, modis_x), aod_clean,
            bounds_error=False, fill_value=0.0
        )
        lon_gr, lat_gr = np.meshgrid(lons_tile, lats_tile)
        pts = np.stack([lat_gr.ravel(), lon_gr.ravel()], axis=1)
        aod_5x5 = interp(pts).reshape(5, 5)
        aod_vals.append(float(np.nanmean(aod_5x5)))
    except Exception:
        aod_vals.append(0.0)

df["modis_aod_055"] = aod_vals
nz = int((df["modis_aod_055"] != 0).sum())
print(f"  OK: MODIS AOD, nonzero={nz}/{len(df)}")

# ============================================================
# 4. Reconstruir secuencias e inyectar
# ============================================================
print()
print("  [4] Reconstruyendo secuencias...")
df["celda_lat"] = (df["centroide_lat"] / CELL_RES).round() * CELL_RES
df["celda_lon"] = (df["centroide_lon"] / CELL_RES).round() * CELL_RES

sequences = []
grouped = df.groupby(["celda_lat", "celda_lon"])
for (clat, clon), grp in grouped:
    ord_grp = grp.sort_values("fecha_dt")
    fechas_u = ord_grp["fecha_dt"].unique()
    if len(fechas_u) < SEQ_LEN:
        continue
    for i in range(len(fechas_u) - SEQ_LEN + 1):
        ventana = fechas_u[i : i + SEQ_LEN]
        rows = [ord_grp[ord_grp["fecha_dt"] == f].iloc[0] for f in ventana]
        sequences.append(rows)
print(f"  Secuencias: {len(sequences)}")
assert len(sequences) == len(X_existing), f"Mismatch! {len(sequences)} vs {len(X_existing)}"

# Inyectar canales
print()
print("  [5] Inyectando covariables (522 -> 531 canales)...")
X_extra = np.zeros((len(sequences), SEQ_LEN, N_COV, 5, 5), dtype=np.float32)
for si, rows in enumerate(tqdm(sequences, desc="Inyectando")):
    for ti in range(SEQ_LEN):
        row = rows[ti]
        for ci, col in enumerate(COV_CHANNELS):
            val = row.get(col, 0.0)
            X_extra[si, ti, ci, :, :] = val if np.isfinite(val) else 0.0

X_full = X_existing[:]
X_new = np.concatenate([X_full, X_extra], axis=2)
print(f"  X: {X_full.shape} -> {X_new.shape}")

# Stats
print()
print("  Stats de canales extra:")
for ci, col in enumerate(COV_CHANNELS):
    vals = X_extra[:, :, ci, :, :]
    nz = int(np.count_nonzero(vals))
    m = float(vals.mean())
    s = float(vals.std())
    print(f"    Canal {522+ci} {col:25s}: nonzero={nz:6d}/{vals.size}  mean={m:8.4f} +- {s:8.4f}")

# Guardar
OUT_DIR.mkdir(exist_ok=True)
np.save(OUT_DIR / "X_convlstm.npy", X_new)
np.save(OUT_DIR / "y_convlstm.npy", np.load(SIT3_DIR / "y_convlstm.npy"))
print(f"\n  [OK] Tensor guardado: {OUT_DIR}/X_convlstm.npy ({X_new.shape})")
print(f"  [OK] Tensor guardado: {OUT_DIR}/y_convlstm.npy")
print("=" * 60)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 104.6 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.
  === Contenido de /kaggle/input/ ===
  Dir: datasets/
    samuelpatino/dagma-data/DAGMA_con_Acopi_NO2.parquet (2405 KB)
    samuelpatino/geovision-cali-sit2/grid_embeddings_sit2.npz (105132 KB)
    samuelpatino/geovision-cali-sit2/metadatos.parquet (327 KB)
  DATA_DIR = /kaggle/input/datasets/samuelpatino/geovision-cali-sit2 (Kaggle rglob)
  SIT3_DIR = /content/dataset_sit3 (se descargara de HF)

  Verificando rutas:
    metadatos.parquet en DATA_DIR: True
    X_convlstm.npy en SIT3_DIR:   False
  [0] Descargando paneles ERA5 y MODIS...
  ERA-5: descargando desde HuggingFace...


Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

  MODIS_MCD: descargando desde HuggingFace...


Fetching 64 files:   0%|          | 0/64 [00:00<?, ?it/s]

  OK

  Tensor no encontrado en Kaggle, descargando desde HuggingFace...


...

  OK: 15 columnas ERA5

  [3] Extrayendo MODIS AOD...


MODIS:   0%|          | 0/2263 [00:00<?, ?it/s]

  OK: MODIS AOD, nonzero=0/2263

  [4] Reconstruyendo secuencias...
  Secuencias: 1403

  [5] Inyectando covariables (522 -> 531 canales)...


Inyectando:   0%|          | 0/1403 [00:00<?, ?it/s]

  X: (1403, 8, 531, 5, 5) -> (1403, 8, 540, 5, 5)

  Stats de canales extra:
    Canal 522 era5_t2m                 : nonzero=280600/280600  mean=  0.5174 +-   0.3313
    Canal 523 era5_blh                 : nonzero=280600/280600  mean=  0.1224 +-   0.0453
    Canal 524 era5_wind_speed          : nonzero=280600/280600  mean=  0.2609 +-   0.0977
    Canal 525 era5_sp                  : nonzero=280600/280600  mean= -0.3073 +-   0.9836
    Canal 526 era5_tp                  : nonzero=274425/280600  mean=  0.5627 +-   1.2951
    Canal 527 modis_aod_055            : nonzero=     0/280600  mean=  0.0000 +-   0.0000
    Canal 528 era5_t2m_delta           : nonzero=176125/280600  mean= -0.0006 +-   0.3060
    Canal 529 era5_blh_delta           : nonzero=176325/280600  mean=  0.0012 +-   0.0457
    Canal 530 era5_wind_speed_delta    : nonzero=176325/280600  mean=  0.0032 +-   0.1021

  [OK] Tensor guardado: /content/dataset_sit3_extra/X_convlstm.npy ((1403, 8, 540, 5, 5))
  [OK] Tensor guardado

# Sit.3 — Entrenar Conv3D (CNN 3D) + XGBoost

Predice NO2, SO2, O3 en T+1, T+3, T+7 a partir de secuencias
de 8 embeddings CLIP (generados por sit2_sae_posttrain.ipynb).

Arquitectura Conv3D:
  Input (B, 8, 522, 5, 5) -> Conv3D(32->64->128) -> Pool -> 3 cabezas separadas
Arquitectura XGBoost:
  Pool espacial + flatten -> 3 XGBoost por contaminante con MultiOutputRegressor

Loss: MSE enmascarada con pesos [1, 1, 1] (datos normalizados con z-score train-only)
Dataset: Slucu-0310/geovision-cali-sit3


In [4]:
# @title Setup: crear modulos src/ localmente
import sys, os, shutil
sys.path.insert(0, ".")

# Limpiar modulos antiguos
for old_file in ["src/modelos/convlstm.py", "src/training/lit_convlstm.py"]:
    if os.path.exists(old_file):
        os.remove(old_file)
for cache_dir in ["src/__pycache__", "src/modelos/__pycache__", "src/training/__pycache__"]:
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir, ignore_errors=True)

os.makedirs("src/modelos", exist_ok=True)
os.makedirs("src/training", exist_ok=True)


# ===== conv3d_sit3.py =====
with open("src/modelos/conv3d_sit3.py", "w") as f:
    f.write("""\
import torch
import torch.nn as nn
import torch.nn.functional as F


class Conv3DSit3(nn.Module):
    '''3D CNN para series de tiempo espacio-temporales (Situacion 3).

    Arquitectura:
      Input: (B, T=8, C=522, H=5, W=5)
      -> Permute a (B, C, T, H, W)
      -> Bottleneck 1x1x1: 522 -> 32
      -> Conv3D(32->64, k=3) + BN + ReLU + MaxPool3d(2)
      -> Conv3D(64->128, k=3) + BN + ReLU + MaxPool3d(2)
      -> AdaptiveAvgPool3d(1) -> Flatten -> (B, 128)
      -> Dropout
      -> 3 cabezas lineales separadas (una por contaminante)
      -> Output: (B, 3 horizontes, 3 contaminantes)
    '''
    def __init__(self, input_channels=522, num_timesteps=8,
                 hidden_dim=64, num_horizons=3, num_pollutants=3,
                 dropout_prob=0.3):
        super().__init__()
        self.num_horizons = num_horizons
        self.num_pollutants = num_pollutants

        # Bottleneck: reducir 522 canales a 32
        self.bottleneck = nn.Conv3d(input_channels, 32, kernel_size=1)
        self.bn_bottleneck = nn.BatchNorm3d(32)

        # Bloque 1: (B, 32, 8, 5, 5) -> (B, 64, 4, 3, 3)
        self.conv1 = nn.Conv3d(32, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.bn1 = nn.BatchNorm3d(64)
        self.pool1 = nn.MaxPool3d(kernel_size=(2, 2, 2))

        # Bloque 2: (B, 64, 4, 3, 3) -> (B, 128, 2, 2, 2)
        self.conv2 = nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.bn2 = nn.BatchNorm3d(128)
        self.pool2 = nn.MaxPool3d(kernel_size=(2, 2, 2))

        # Pooling global y dropout
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.dropout = nn.Dropout(dropout_prob)

        # Cabezas separadas por contaminante
        # Cada cabeza: 128 features -> 3 horizontes
        self.heads = nn.ModuleList([
            nn.Linear(128, num_horizons) for _ in range(num_pollutants)
        ])

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv3d, nn.Linear)):
                nn.init.xavier_uniform_(m.weight, gain=0.5)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        '''Forward pass.

        Args:
            x: (B, T, C, H, W) con T=8 timesteps, C=522 canales, H=W=5
        Returns:
            (B, num_horizons, num_pollutants)
        '''
        # Permute: (B, T, C, H, W) -> (B, C, T, H, W) para Conv3d
        x = x.permute(0, 2, 1, 3, 4)

        # Bottleneck
        x = self.bottleneck(x)
        x = self.bn_bottleneck(x)
        x = F.relu(x)

        # Bloque 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)

        # Bloque 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)

        # Pooling global
        x = self.global_pool(x)          # (B, 128, 1, 1, 1)
        x = x.flatten(1)                  # (B, 128)
        x = self.dropout(x)

        # Cabezas separadas
        outs = [head(x) for head in self.heads]  # cada una: (B, 3)

        # Stack: (B, num_horizons, num_pollutants)
        return torch.stack(outs, dim=-1)
""")
print("  Creado: src/modelos/conv3d_sit3.py (97 lines)")

# ===== lit_conv3d.py =====
with open("src/training/lit_conv3d.py", "w") as f:
    f.write("""\
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl
from src.modelos.conv3d_sit3 import Conv3DSit3


class Sit3Conv3DDataset(Dataset):
    'Dataset que carga X e y desde archivos .npy.'
    def __init__(self, x_path, y_path, split_indices=None):
        self.X = np.load(x_path, mmap_mode="r")
        self.y = np.load(y_path, mmap_mode="r")
        if split_indices is not None:
            self.X = self.X[split_indices]
            self.y = self.y[split_indices]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx].copy()).float()
        y = torch.from_numpy(self.y[idx].copy()).float()
        return {"x": x, "y": y}


POLLUTANTS = ["NO2", "SO2", "O3"]
HORIZONS = ["T+1", "T+3", "T+7"]
CONV_FACTOR = {"NO2": 5750.0, "SO2": 8008.0, "O3": 6000.0}
KPI_UGM3 = {"NO2": 8.0, "SO2": 6.0, "O3": 12.0}


class LitConv3D(pl.LightningModule):
    'LightningModule para Conv3DSit3 (Situacion 3).'
    WEIGHTS_DEFAULT = [1.0, 1.0, 1.0]

    def __init__(self, model, lr=1e-4, weight_decay=1e-5, loss_weights=None,
                 y_mean=None, y_std=None):
        super().__init__()
        self.save_hyperparameters(ignore=["model"])
        self.model = model
        self.lr = lr
        self.weight_decay = weight_decay
        self.loss_weights = loss_weights or self.WEIGHTS_DEFAULT
        self.y_mean = y_mean
        self.y_std = y_std
        self._test_preds = []
        self._test_trues = []

    def forward(self, x):
        return self.model(x)

    def _loss(self, y_pred, y_true):
        'MSE enmascarada con pesos por contaminante.'
        w = torch.tensor(self.loss_weights, device=y_pred.device)
        mask = ~torch.isnan(y_true)
        loss_p = torch.zeros(3, device=y_pred.device)
        for p in range(3):
            pmask = mask[:, :, p]
            if pmask.sum() > 0:
                loss_p[p] = F.mse_loss(y_pred[:, :, p][pmask], y_true[:, :, p][pmask], reduction="mean")
        return (loss_p * w).sum() / w.sum()

    def training_step(self, batch, batch_idx):
        y_pred = self.model(batch["x"])
        loss = self._loss(y_pred, batch["y"])
        self.log("train/loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        y_pred = self.model(batch["x"])
        loss = self._loss(y_pred, batch["y"])
        self.log("val/loss", loss, on_epoch=True, prog_bar=True)

    def test_step(self, batch, batch_idx):
        y_pred = self.model(batch["x"])
        y_true = batch["y"]
        self.log("test/loss", self._loss(y_pred, y_true), on_epoch=True)
        self._test_preds.append(y_pred.detach().cpu())
        self._test_trues.append(y_true.detach().cpu())

    def on_test_end(self):
        'Imprime tabla de metricas completa al final del test.'
        all_preds = torch.cat(self._test_preds, dim=0).numpy()
        all_trues = torch.cat(self._test_trues, dim=0).numpy()
        # Denormalizar si stats disponibles
        if self.y_mean is not None and self.y_std is not None:
            ym = np.array(self.y_mean).reshape(1, 1, 3)
            ys = np.array(self.y_std).reshape(1, 1, 3)
            all_preds = all_preds * ys + ym
            all_trues = all_trues * ys + ym
        mask = ~np.isnan(all_trues)

        print()
        print("=" * 70)
        print("  TABLA RESUMEN - Conv3D (SIT. 3)")
        print("=" * 70)

        # --- Global ---
        loss_val = float(np.nanmean((all_preds - all_trues) ** 2))
        rmse_val = float(np.sqrt(loss_val))
        print(f"  MSE global:              {loss_val:.6e}")
        print(f"  RMSE global (mol/m2):    {rmse_val:.6f}")
        ss_res_g = float(np.nansum((all_preds - all_trues) ** 2))
        ss_tot_g = float(np.nansum((all_trues - np.nanmean(all_trues)) ** 2))
        r2_g = 1.0 - ss_res_g / (ss_tot_g + 1e-12) if ss_tot_g > 0 else float("nan")
        print(f"  R2 global:               {r2_g:.6f}")
        print()

        # --- Por contaminante ---
        print(f"  {'Contaminante':15s} {'RMSE mol/m2':14s} {'RMSE ug/m3':12s} {'KPI ug/m3':10s} {'Cumple':8s} {'R2':8s}")
        print(f"  {'-'*15} {'-'*14} {'-'*12} {'-'*10} {'-'*8} {'-'*8}")
        for pi, pn in enumerate(POLLUTANTS):
            pmask = mask[:, :, pi]
            y_p = all_preds[:, :, pi][pmask]
            y_t = all_trues[:, :, pi][pmask]
            if len(y_p) > 0:
                rmse_p = float(np.sqrt(np.mean((y_p - y_t) ** 2)))
                ug = rmse_p * CONV_FACTOR.get(pn, 1.0)
                ss_res = float(np.sum((y_p - y_t) ** 2))
                ss_tot = float(np.sum((y_t - np.mean(y_t)) ** 2))
                r2 = 1.0 - ss_res / (ss_tot + 1e-12) if ss_tot > 0 else float("nan")
                kpi = KPI_UGM3.get(pn, float("inf"))
                cumple = "SI" if ug <= kpi else "NO"
            else:
                rmse_p, ug, r2, cumple = float("nan"), float("nan"), float("nan"), "N/A"
            print(f"  {pn:15s} {rmse_p:>8.2e}     {ug:>7.2f} ug/m3  {kpi:>6.1f}     {cumple:8s}  {r2:>7.4f}")

        print()

        # --- Por horizonte (CADA contaminante por separado) ---
        print("  R2 por horizonte y contaminante:")
        print(f"  {'Horizonte':10s} {'NO2 R2':8s} {'SO2 R2':8s} {'O3 R2':8s}")
        print(f"  {'-'*10} {'-'*8} {'-'*8} {'-'*8}")
        for hi, hn in enumerate(HORIZONS):
            r2s = []
            for pi in range(3):
                m = mask[:, hi, pi]
                y_ph = all_preds[:, hi, pi][m]
                y_th = all_trues[:, hi, pi][m]
                if len(y_ph) > 0:
                    ss_res = float(np.sum((y_ph - y_th) ** 2))
                    ss_tot = float(np.sum((y_th - np.mean(y_th)) ** 2))
                    r2_h = 1.0 - ss_res / (ss_tot + 1e-12) if ss_tot > 0 else float("nan")
                else:
                    r2_h = float("nan")
                r2s.append(r2_h)
            print(f"  {hn:10s} {r2s[0]:>7.4f}  {r2s[1]:>7.4f}  {r2s[2]:>7.4f}")

        # --- Degradacion por contaminante ---
        print()
        print("  Degradacion T+1 -> T+7 por contaminante:")
        for pi, pn in enumerate(POLLUTANTS):
            m1 = mask[:, 0, pi]
            m7 = mask[:, 2, pi]
            if m1.sum() > 0 and m7.sum() > 0:
                r1 = float(np.sqrt(np.mean((all_preds[:, 0, pi][m1] - all_trues[:, 0, pi][m1]) ** 2)))
                r7 = float(np.sqrt(np.mean((all_preds[:, 2, pi][m7] - all_trues[:, 2, pi][m7]) ** 2)))
                degradacion = (r7 / r1 - 1) * 100 if r1 > 0 else float("nan")
            else:
                degradacion = float("nan")
            print(f"    {pn:4s}: {degradacion:.1f}%" if not np.isnan(degradacion) else f"    {pn:4s}: N/A")

        print()
        print("=" * 70)
        self._test_preds.clear()
        self._test_trues.clear()

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=10, min_lr=1e-7)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "monitor": "val/loss"}}
""")
print("  Creado: src/training/lit_conv3d.py (171 lines)")

print("Modulos src/ creados localmente")
print("OK: Conv3DSit3 con ~295K parametros")
print("OK: LitConv3D con tabla de metricas corregida")


  Creado: src/modelos/conv3d_sit3.py (97 lines)
  Creado: src/training/lit_conv3d.py (171 lines)
Modulos src/ creados localmente
OK: Conv3DSit3 con ~295K parametros
OK: LitConv3D con tabla de metricas corregida


## 1. Descargar dataset de HuggingFace

Dataset: Slucu-0310/geovision-cali-sit3 (X_convlstm.npy + y_convlstm.npy)


In [5]:
# @title Descargar dataset
from huggingface_hub import snapshot_download
import shutil

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

DATA_DIR = Path("/content/dataset_sit3")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Si ya existe, no hacer nada
if (DATA_DIR / "X_convlstm.npy").is_file():
    print(f"Dataset ya existe en {DATA_DIR}")
else:
    # 1. Buscar en Kaggle input primero
    kaggle_src = None
    if Path("/kaggle/input").exists():
        for d in sorted(Path("/kaggle/input").iterdir()):
            if (d / "X_convlstm.npy").exists() and (d / "y_convlstm.npy").exists():
                kaggle_src = d
                break
    
    if kaggle_src:
        # Copiar de Kaggle a /content/ (porque /kaggle/input/ es read-only)
        print(f"Copiando dataset desde Kaggle: {kaggle_src.name}")
        shutil.copy(kaggle_src / "X_convlstm.npy", DATA_DIR / "X_convlstm.npy")
        shutil.copy(kaggle_src / "y_convlstm.npy", DATA_DIR / "y_convlstm.npy")
        # Tambien copiar otros archivos utiles
        for f in kaggle_src.iterdir():
            if f.suffix in ('.npy', '.npz', '.parquet'):
                try:
                    shutil.copy(f, DATA_DIR / f.name)
                except:
                    pass
    else:
        # 2. Descargar de HuggingFace
        print("Descargando dataset desde HuggingFace...")
        snapshot_download(
            repo_id="Slucu-0310/geovision-cali-sit3",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            token=HF_TOKEN,
        )
    
    print(f"Dataset listo en {DATA_DIR}")
    # Listar archivos
    for f in sorted(DATA_DIR.iterdir()):
        size_mb = f.stat().st_size / (1024*1024) if f.is_file() else 0
        if size_mb > 0:
            print(f"  {f.name}: {size_mb:.1f} MB")


Dataset ya existe en /content/dataset_sit3


## 2. Cargar datos y crear splits

Split 70/15/15. Normalizacion z-score SOLO con estadisticas de train.
Se guarda copia normalizada (y_convlstm_norm.npy) para entrenar.


In [6]:
# @title Cargar X, y y crear splits (HONESTOS, sin leakage de ventanas)
# Las secuencias se construyeron con ventana deslizante de 8 timesteps sobre tiles
# que comparten ubicacion. Un split aleatorio mete secuencias casi clonadas en
# train/test (memorizacion). Aqui se usan BLOQUES CONTIGUOS del tensor como proxy
# de tile_id y se deja un gap entre splits para evitar solapamiento de ventanas.
#
# Si quieres aun mas rigor, alimenta SPLIT_BY_GROUP con tile_id (1 entero por
# secuencia) y usa GroupShuffleSplit; el codigo lo soporta abajo.

X = np.load(DATA_DIR / "X_convlstm.npy", mmap_mode="r")
y_full = np.load(DATA_DIR / "y_convlstm.npy", mmap_mode="r")
print(f"X: {X.shape}  y: {y_full.shape}")

N = len(X)
WINDOW_GAP = 8  # buffer entre bloques (igual al WINDOW para evitar solapamiento)

# ---- A) Split por GRUPOS si tenemos tile_id (preferido) ----
SPLIT_BY_GROUP = None  # array (N,) con tile_id por secuencia, o None
# Intentar inferir desde metadatos si estan en este Kaggle input
try:
    candidates = [
        Path("/kaggle/input/datasets/samuelpatino/geovision-cali-sit2/metadatos.parquet"),
        DATA_DIR / "metadatos.parquet",
    ]
    meta_p = next((p for p in candidates if p.exists()), None)
    if meta_p is not None:
        _meta = pd.read_parquet(meta_p)
        if {"centroide_lat", "centroide_lon"}.issubset(_meta.columns) and len(_meta) >= N:
            keys = (np.round(_meta["centroide_lat"].values[:N], 4).astype(str) + "_" +
                    np.round(_meta["centroide_lon"].values[:N], 4).astype(str))
            _, group_ids = np.unique(keys, return_inverse=True)
            SPLIT_BY_GROUP = group_ids
            print(f"  Split por GRUPO: {len(np.unique(group_ids))} tiles distintos")
except Exception as e:
    print(f"  (info) no se pudo inferir tile_id desde metadatos: {e}")

if SPLIT_BY_GROUP is not None:
    from sklearn.model_selection import GroupShuffleSplit
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    tr, tmp = next(gss1.split(np.arange(N), groups=SPLIT_BY_GROUP))
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    va_rel, te_rel = next(gss2.split(tmp, groups=SPLIT_BY_GROUP[tmp]))
    train_idx, val_idx, test_idx = tr, tmp[va_rel], tmp[te_rel]
    split_kind = "group (tile_id)"
else:
    # ---- B) Fallback: bloques contiguos + gap (sin solapamiento de ventanas) ----
    cut_tr = int(0.70 * N)
    cut_va = int(0.85 * N)
    train_idx = np.arange(0, max(0, cut_tr - WINDOW_GAP))
    val_idx   = np.arange(cut_tr, max(cut_tr, cut_va - WINDOW_GAP))
    test_idx  = np.arange(cut_va, N)
    split_kind = f"bloques contiguos (gap={WINDOW_GAP})"

print(f"Split {split_kind} -> Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

# Normalizar y con z-score usando SOLO estadisticas de TRAIN
y_train_vals = y_full[train_idx]
y_mean = np.nanmean(y_train_vals, axis=(0, 1), keepdims=True)  # (1,1,3)
y_std = np.nanstd(y_train_vals, axis=(0, 1), keepdims=True)    # (1,1,3)
y_std[y_std < 1e-12] = 1.0
print(f"Normalizacion z-score (TRAIN only): mean={np.squeeze(y_mean)}, std={np.squeeze(y_std)}")

# Normalizar TODO el dataset con las stats de train
y_data = y_full[:]
y_norm = (y_data - y_mean) / y_std
y_norm = np.where(np.isnan(y_data), np.nan, y_norm)

# Guardar y normalizado a archivo temporal
y_norm_path = DATA_DIR / "y_convlstm_norm.npy"
np.save(y_norm_path, y_norm.astype(np.float32))
print(f"y normalizado guardado en: {y_norm_path}")

Y_NORM_PATH = str(y_norm_path)


X: (1403, 8, 531, 5, 5)  y: (1403, 3, 3)
  Split por GRUPO: 1318 tiles distintos
Split group (tile_id) -> Train: 980, Val: 209, Test: 214
Normalizacion z-score (TRAIN only): mean=[2.95467071e-05 2.62571586e-04 1.15940906e-01], std=[1.8220788e-05 2.4828501e-04 5.5678594e-03]
y normalizado guardado en: /content/dataset_sit3/y_convlstm_norm.npy


In [7]:
# @title Crear datasets y dataloaders
from lightning.pytorch.loggers import CSVLogger
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl

BATCH_SIZE = 32



## 3. Modelos

- Conv3D: bottleneck 522->32, 2 bloques Conv3D con BN, pooling global, cabezas separadas (~295K params)
- XGBoost: features planas (pool 5x5 + flatten a 4176), un modelo por contaminante


In [8]:
# @title Crear modelo 3D CNN
import sys
sys.path.insert(0, ".")
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.modelos.conv3d_sit3 import Conv3DSit3

# Detectar canales reales del tensor
actual_channels = int(np.load(DATA_DIR / "X_convlstm.npy", mmap_mode="r").shape[2])
print(f"Tensor X canales detectados: {actual_channels}")

# Modelo 3D CNN con ~295K parametros
model = Conv3DSit3(input_channels=actual_channels)
total_params = sum(p.numel() for p in model.parameters())
print(f"Modelo Conv3DSit3 creado. Parametros: {total_params:,}")

# Stats de normalizacion (calculados en TRAIN only desde celda 6)
Y_MEAN = y_mean
Y_STD = y_std

# Datasets apuntando al Y normalizado
from src.training.lit_conv3d import Sit3Conv3DDataset
ds_train = Sit3Conv3DDataset(str(DATA_DIR / "X_convlstm.npy"), Y_NORM_PATH, split_indices=train_idx.tolist())
ds_val   = Sit3Conv3DDataset(str(DATA_DIR / "X_convlstm.npy"), Y_NORM_PATH, split_indices=val_idx.tolist())
ds_test  = Sit3Conv3DDataset(str(DATA_DIR / "X_convlstm.npy"), Y_NORM_PATH, split_indices=test_idx.tolist())

BATCH_SIZE = 32
dl_train = DataLoader(ds_train, BATCH_SIZE, shuffle=True, num_workers=0)
dl_val   = DataLoader(ds_val,   BATCH_SIZE, shuffle=False, num_workers=0)
dl_test  = DataLoader(ds_test,  BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Batches: train={len(dl_train)}, val={len(dl_val)}, test={len(dl_test)}")

Tensor X canales detectados: 531
Modelo Conv3DSit3 creado. Parametros: 295,305
Batches: train=31, val=7, test=7


In [9]:
# @title Entrenar
from pathlib import Path
from lightning.pytorch.loggers import CSVLogger
import lightning.pytorch as pl

from src.training.lit_conv3d import LitConv3D

RUN_DIR = Path("/content/runs/sit3_conv3d")
RUN_DIR.mkdir(parents=True, exist_ok=True)

lit_model = LitConv3D(model, lr=5e-5, weight_decay=1e-5,
                      y_mean=Y_MEAN, y_std=Y_STD)

callbacks = [
    pl.callbacks.EarlyStopping(monitor="val/loss", patience=40, mode="min"),
    pl.callbacks.ModelCheckpoint(
        dirpath=str(RUN_DIR), filename="best",
        monitor="val/loss", mode="min", save_top_k=1
    ),
]

trainer = pl.Trainer(
    max_epochs=400, accelerator="auto", devices=1,
    callbacks=callbacks,
    logger=CSVLogger(save_dir=str(RUN_DIR / "lightning_logs")),
    log_every_n_steps=5, gradient_clip_val=1.0,
    default_root_dir=str(RUN_DIR),
)

trainer.fit(lit_model, dl_train, dl_val)
print(f"Entrenamiento completo. Mejor checkpoint en {RUN_DIR}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ Conv3DSit3 │  295 K │ train │     0 │
└───┴───────┴────────────┴────────┴───────┴───────┘

Trainable params: 295 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 295 K                                                                                                
Total estimated model params size (MB): 1.181                                                                      
Modules in train mode: 15                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Entrenamiento completo. Mejor checkpoint en /content/runs/sit3_conv3d


## 4. Evaluacion en test



In [10]:
# @title Evaluar en test
import os, glob as gl

# Buscar el mejor checkpoint
ckpts = sorted(RUN_DIR.glob("best*"), key=os.path.getmtime, reverse=True)
if ckpts:
    ckpt_path = str(ckpts[0])
    print(f"Usando checkpoint: {ckpt_path}")
    trainer.test(lit_model, dl_test, ckpt_path=ckpt_path, weights_only=False)
else:
    print("Checkpoint no encontrado, evaluando con ultimo modelo")
    trainer.test(lit_model, dl_test, weights_only=False)

# --- Curvas de entrenamiento ---
csv_files = gl.glob(str(RUN_DIR / "lightning_logs" / "version_*" / "metrics.csv"))
if csv_files:
    import pandas as pd
    import matplotlib.pyplot as plt

    df_log = pd.read_csv(csv_files[0])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    if "train/loss_epoch" in df_log.columns:
        axes[0].plot(df_log["train/loss_epoch"], label="train", alpha=0.7)
    if "val/loss" in df_log.columns:
        axes[0].plot(df_log["val/loss"], label="val", alpha=0.7)
    axes[0].set_xlabel("Epoca")
    axes[0].set_ylabel("MSE Loss")
    axes[0].set_title("Loss de entrenamiento")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # RMSE
    if "val/rmse" in df_log.columns:
        axes[1].plot(df_log["val/rmse"], label="val RMSE", alpha=0.7)
    axes[1].set_xlabel("Epoca")
    axes[1].set_ylabel("RMSE")
    axes[1].set_title("RMSE global (val)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(str(RUN_DIR / "training_curves.png"), dpi=150)
    plt.show()
    print(f"Curvas guardadas en {RUN_DIR / 'training_curves.png'}")
else:
    print("No se encontraron logs de entrenamiento.")


Restoring states from the checkpoint path at /content/runs/sit3_conv3d/best.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /content/runs/sit3_conv3d/best.ckpt


Output()

======================================================================

TABLA RESUMEN - Conv3D (SIT. 3)

======================================================================

...

O3  : -2.1%

======================================================================

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/loss         │    0.8004981875419617     │
└───────────────────────────┴───────────────────────────┘

Usando checkpoint: /content/runs/sit3_conv3d/best.ckpt


No se encontraron logs de entrenamiento.


In [12]:
# @title Guardar pesos del modelo y descargar .zip
import zipfile, hashlib, json
from pathlib import Path
from IPython.display import FileLink, display

OUTPUT_ZIP = "/kaggle/working/sit3_convlstm_weights.zip"
RUN_DIR = Path("/content/runs/sit3_conv3d")

# Buscar el mejor checkpoint
ckpts = list(RUN_DIR.glob("best*"))
if not ckpts:
    print("No checkpoint found. Train first.")
else:
    ckpt_path = str(ckpts[0])
    print(f"Usando checkpoint: {ckpt_path}")

    # Crear .zip con pesos, metadatos y resumen
    with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
        # Checkpoint del modelo
        zf.write(ckpt_path, arcname="best.ckpt")

        # Hash SHA256
        sha = hashlib.sha256()
        with open(ckpt_path, "rb") as fckpt:
            for chunk in iter(lambda: fckpt.read(65536), b""):
                sha.update(chunk)
        zf.writestr("best.ckpt.sha256", sha.hexdigest())

        # Metadata del modelo
        meta = {
            "model": "ConvLSTM2D",
            "input_channels": 522,
            "hidden_dim": 128,
            "num_layers": 3,
            "dropout_prob": 0.3,
            "loss_weights": [1.0, 1.0, 1.0],
            "checkpoint": "best.ckpt",
            "sha256": sha.hexdigest(),
        }
        zf.writestr("metadata.json", json.dumps(meta, indent=2))

        # Curvas de entrenamiento si existen
        curves = RUN_DIR / "training_curves.png"
        if curves.is_file():
            zf.write(str(curves), arcname="training_curves.png")

        # Logs CSV
        log_dir = RUN_DIR / "lightning_logs"
        if log_dir.is_dir():
            for csv_file in log_dir.rglob("*.csv"):
                zf.write(str(csv_file), arcname=f"logs/{csv_file.name}")

    tamano = Path(OUTPUT_ZIP).stat().st_size
    print(f"\n.zip creado: {OUTPUT_ZIP}")
    print(f"Tamaño: {tamano / (1024*1024):.2f} MB")
    print("\nDescarga el archivo haciendo clic en el link de abajo:")
    display(FileLink(OUTPUT_ZIP))


Usando checkpoint: /content/runs/sit3_conv3d/best.ckpt

.zip creado: /kaggle/working/sit3_convlstm_weights.zip
Tamaño: 3.13 MB

Descarga el archivo haciendo clic en el link de abajo:


/kaggle/working/sit3_convlstm_weights.zip

## 5. Curvas de entrenamiento



In [13]:
# @title Curvas de entrenamiento (imagen consolidada)
# Genera training_curves.png con: loss train/val, RMSE por contaminante, LR,
# best epoch marcado y resumen numerico. Pensado para incluir en el informe.

import glob as gl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

csv_files = gl.glob(str(RUN_DIR / "lightning_logs" / "version_*" / "metrics.csv"))
if not csv_files:
    print("No se encontraron logs (lightning_logs/version_*/metrics.csv). Entrena primero.")
else:
    df = pd.read_csv(sorted(csv_files)[-1])
    if "epoch" not in df.columns:
        df["epoch"] = np.arange(len(df))

    # ---- agregar por epoca (Lightning loggea por step y por epoch) ----
    def by_epoch(col):
        if col not in df.columns:
            return None
        s = df[["epoch", col]].dropna()
        if s.empty:
            return None
        return s.groupby("epoch")[col].mean()

    train_loss = by_epoch("train/loss_epoch") or by_epoch("train/loss")
    val_loss   = by_epoch("val/loss")
    val_rmse_no2 = by_epoch("val/rmse/NO2") or by_epoch("val/rmse_NO2")
    val_rmse_so2 = by_epoch("val/rmse/SO2") or by_epoch("val/rmse_SO2")
    val_rmse_o3  = by_epoch("val/rmse/O3")  or by_epoch("val/rmse_O3")
    val_rmse_global = by_epoch("val/rmse")
    lr_series = by_epoch("lr-AdamW") or by_epoch("lr") or by_epoch("learning_rate")

    best_epoch = int(val_loss.idxmin()) if val_loss is not None else None
    best_val   = float(val_loss.min()) if val_loss is not None else float("nan")

    # ---- metricas test (ultima fila con datos test/*) ----
    test_cols = [c for c in df.columns if c.startswith("test/")]
    test_summary = {}
    if test_cols:
        tail = df[test_cols].dropna(how="all")
        if not tail.empty:
            last = tail.iloc[-1]
            test_summary = {c: float(last[c]) for c in test_cols if pd.notna(last[c])}

    # ---- figura 2x2 ----
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("Sit.3 Conv3D — Curvas de entrenamiento", fontsize=14, y=0.995)

    ax = axes[0, 0]
    if train_loss is not None:
        ax.plot(train_loss.index, train_loss.values, label="train", linewidth=1.5)
    if val_loss is not None:
        ax.plot(val_loss.index, val_loss.values, label="val", linewidth=1.5)
        if best_epoch is not None:
            ax.axvline(best_epoch, color="green", linestyle="--", alpha=0.6,
                       label=f"best (ep {best_epoch}, val={best_val:.3f})")
    ax.set_title("Loss (MSE normalizada)")
    ax.set_xlabel("Epoca"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.set_yscale("log")

    ax = axes[0, 1]
    for s, name, color in [
        (val_rmse_no2, "NO2", "tab:blue"),
        (val_rmse_so2, "SO2", "tab:orange"),
        (val_rmse_o3,  "O3",  "tab:green"),
    ]:
        if s is not None:
            ax.plot(s.index, s.values, label=name, color=color, linewidth=1.5)
    if val_rmse_global is not None:
        ax.plot(val_rmse_global.index, val_rmse_global.values,
                color="black", linestyle=":", alpha=0.7, label="global")
    # Lineas KPI
    for thr, name, color in [(8, "NO2 KPI", "tab:blue"),
                              (6, "SO2 KPI", "tab:orange"),
                              (12, "O3 KPI", "tab:green")]:
        ax.axhline(thr, color=color, linestyle="--", alpha=0.4, linewidth=0.8)
    ax.set_title("RMSE por contaminante (val) — lineas punteadas: KPI")
    ax.set_xlabel("Epoca"); ax.set_ylabel("RMSE (ug/m3)")
    ax.legend(loc="best", fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    if lr_series is not None:
        ax.plot(lr_series.index, lr_series.values, color="purple", linewidth=1.5)
        ax.set_yscale("log")
    ax.set_title("Learning rate")
    ax.set_xlabel("Epoca"); ax.set_ylabel("lr")
    ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    ax.axis("off")
    lines = ["Resumen final"]
    lines.append("-" * 32)
    if best_epoch is not None:
        lines.append(f"Best epoch (val): {best_epoch}")
        lines.append(f"Best val loss:    {best_val:.4f}")
    if test_summary:
        lines.append("")
        lines.append("Metricas test (split honesto):")
        for k, v in test_summary.items():
            lines.append(f"  {k}: {v:.4f}")
    lines.append("")
    lines.append("Nota: el KPI principal de Sit.3 se evalua")
    lines.append("via LOO-CV vs DAGMA (celdas 24-27).")
    ax.text(0.0, 1.0, "\n".join(lines), family="monospace",
            fontsize=10, va="top", ha="left", transform=ax.transAxes)

    fig.tight_layout()
    out_png = RUN_DIR / "training_curves.png"
    fig.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nCurvas guardadas en {out_png}")
    if test_summary:
        print("Metricas test:")
        for k, v in test_summary.items():
            print(f"  {k}: {v:.4f}")



No se encontraron logs (lightning_logs/version_*/metrics.csv). Entrena primero.


## 6. Resumen de metricas

| KPI | Minimo | Medicion |
|-----|--------|----------|
| RMSE LOO-CV NO2 (T+1) | <= 8 ug/m3 | (test) |
| RMSE LOO-CV SO2 (T+1) | <= 6 ug/m3 | (test) |
| RMSE LOO-CV O3 (T+1) | <= 12 ug/m3 | (test) |
| R2 LOO-CV (promedio) | >= 0.55 | (test, targets absolutos) |
| Degradacion T+1->T+7 | < 60% | (test) |


## 7. XGBoost por contaminante (features planas)

Se aplanan los tensores: pool espacial 5x5 -> media -> (N, 8, 522) -> flatten -> (N, 4176) features.

Se entrena un XGBoost por contaminante con MultiOutputRegressor (3 horizontes).
Hiperparametros: max_depth=6, n_estimators=500, learning_rate=0.05, early_stopping con 50 rounds.

In [14]:
# @title 7a. Feature engineering y entrenamiento XGBoost
import numpy as np
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Cargar datos (si no estan en memoria)
try:
    X_full = X
    y_full = y_full  # from earlier cell
except NameError:
    X_full = np.load(DATA_DIR / "X_convlstm.npy", mmap_mode="r")
    y_full_np = np.load(DATA_DIR / "y_convlstm.npy", mmap_mode="r")

# Feature engineering: pool espacial + flatten
# X shape: (N, 8, 522, 5, 5) -> pool(5,5) -> (N, 8, 522) -> flatten -> (N, 4176)
X_pool = X_full.mean(axis=(-1, -2))  # (N, 8, 522)
X_feat = X_pool.reshape(len(X_pool), -1)  # (N, 4176)
print(f"Features: {X_feat.shape[1]}, Muestras: {len(X_feat)}")

# Cargar y real (sin normalizar) y normalizar con stats de train
y_raw = np.load(DATA_DIR / "y_convlstm.npy", mmap_mode="r")
y_data = y_raw[:]

# Usar y_mean/y_std de celda 6 (train-only) o calcularlos
try:
    ym = y_mean
    ys = y_std
except NameError:
    y_train_vals = y_data[train_idx]
    ym = np.nanmean(y_train_vals, axis=(0, 1), keepdims=True)
    ys = np.nanstd(y_train_vals, axis=(0, 1), keepdims=True)
    ys[ys < 1e-12] = 1.0

y_norm = (y_data - ym) / ys
y_norm = np.where(np.isnan(y_data), np.nan, y_norm)

# Splits
from sklearn.model_selection import train_test_split
try:
    tr_idx = train_idx
    vl_idx = val_idx
    te_idx = test_idx
except NameError:
    N = len(X_feat)
    idx = np.arange(N)
    tr_idx, tmp_idx = train_test_split(idx, test_size=0.3, random_state=42)
    vl_idx, te_idx = train_test_split(tmp_idx, test_size=0.5, random_state=42)

X_tr, X_vl, X_te = X_feat[tr_idx], X_feat[vl_idx], X_feat[te_idx]
print(f"Splits: train={len(X_tr)}, val={len(X_vl)}, test={len(X_te)}")

# Entrenar 3 modelos XGBoost (uno por contaminante)
models = {}
for pi, pn in enumerate(["NO2", "SO2", "O3"]):
    y_tr = y_norm[tr_idx][:, :, pi]  # (N, 3) - 3 horizontes
    y_vl = y_norm[vl_idx][:, :, pi]
    
    # Reemplazar NaNs con 0 para entrenamiento
    y_tr_clean = np.nan_to_num(y_tr, nan=0.0)
    y_vl_clean = np.nan_to_num(y_vl, nan=0.0)
    
    base = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.3,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        verbosity=0,
    )
    model = MultiOutputRegressor(base)
    model.fit(X_tr, y_tr_clean)
    models[pn] = model
    print(f"  {pn}: XGBoost entrenado")

print("Entrenamiento XGBoost completo.")


Features: 4248, Muestras: 1403
Splits: train=980, val=209, test=214
  NO2: XGBoost entrenado
  SO2: XGBoost entrenado
  O3: XGBoost entrenado
Entrenamiento XGBoost completo.


In [15]:
# @title 7b. Evaluacion XGBoost
from sklearn.metrics import mean_squared_error, r2_score

POLLUTANTS = ["NO2", "SO2", "O3"]
HORIZONS = ["T+1", "T+3", "T+7"]
CONV_FACTOR = {"NO2": 5750.0, "SO2": 8008.0, "O3": 6000.0}
KPI_UGM3 = {"NO2": 8.0, "SO2": 6.0, "O3": 12.0}

# Predecir en test
preds_dict = {}
for pi, pn in enumerate(POLLUTANTS):
    pred_norm = models[pn].predict(X_te)  # (N, 3)
    preds_dict[pn] = pred_norm

# Stack predicciones: (N_test, 3, 3)
all_preds = np.stack([preds_dict[pn] for pn in POLLUTANTS], axis=-1)  # (N, 3, 3)
all_trues = y_norm[te_idx]  # (N, 3, 3)

# Denormalizar
ym_1 = np.array(ym).reshape(1, 1, 3)
ys_1 = np.array(ys).reshape(1, 1, 3)
all_preds_denorm = all_preds * ys_1 + ym_1
all_trues_denorm = all_trues * ys_1 + ym_1

mask = ~np.isnan(all_trues_denorm)

print()
print("=" * 70)
print("  TABLA RESUMEN - XGBoost (SIT. 3)")
print("=" * 70)

mse_g = float(np.nanmean((all_preds_denorm - all_trues_denorm) ** 2))
print(f"  MSE global:              {mse_g:.6e}")
print(f"  RMSE global (mol/m2):    {np.sqrt(mse_g):.6f}")
ss_res_g = float(np.nansum((all_preds_denorm - all_trues_denorm) ** 2))
ss_tot_g = float(np.nansum((all_trues_denorm - np.nanmean(all_trues_denorm)) ** 2))
r2_g = 1.0 - ss_res_g / (ss_tot_g + 1e-12) if ss_tot_g > 0 else float("nan")
print(f"  R2 global:               {r2_g:.6f}")
print()

# Por contaminante
print(f"  {'Contaminante':15s} {'RMSE mol/m2':14s} {'RMSE ug/m3':12s} {'KPI ug/m3':10s} {'Cumple':8s} {'R2':8s}")
print(f"  {'-'*15} {'-'*14} {'-'*12} {'-'*10} {'-'*8} {'-'*8}")
for pi, pn in enumerate(POLLUTANTS):
    pmask = mask[:, :, pi]
    y_p = all_preds_denorm[:, :, pi][pmask]
    y_t = all_trues_denorm[:, :, pi][pmask]
    if len(y_p) > 0:
        rmse_p = float(np.sqrt(np.mean((y_p - y_t) ** 2)))
        ug = rmse_p * CONV_FACTOR.get(pn, 1.0)
        ss_res = float(np.sum((y_p - y_t) ** 2))
        ss_tot = float(np.sum((y_t - np.mean(y_t)) ** 2))
        r2 = 1.0 - ss_res / (ss_tot + 1e-12) if ss_tot > 0 else float("nan")
        kpi = KPI_UGM3.get(pn, float("inf"))
        cumple = "SI" if ug <= kpi else "NO"
    else:
        rmse_p, ug, r2, cumple = float("nan"), float("nan"), float("nan"), "N/A"
    print(f"  {pn:15s} {rmse_p:>8.2e}     {ug:>7.2f} ug/m3  {kpi:>6.1f}     {cumple:8s}  {r2:>7.4f}")

print()

# R2 por horizonte y contaminante
print(f"  {'Horizonte':10s} {'NO2 R2':8s} {'SO2 R2':8s} {'O3 R2':8s}")
print(f"  {'-'*10} {'-'*8} {'-'*8} {'-'*8}")
for hi, hn in enumerate(HORIZONS):
    r2s = []
    for pi in range(3):
        m = mask[:, hi, pi]
        y_ph = all_preds_denorm[:, hi, pi][m]
        y_th = all_trues_denorm[:, hi, pi][m]
        if len(y_ph) > 0:
            ss_res = float(np.sum((y_ph - y_th) ** 2))
            ss_tot = float(np.sum((y_th - np.mean(y_th)) ** 2))
            r2_h = 1.0 - ss_res / (ss_tot + 1e-12) if ss_tot > 0 else float("nan")
        else:
            r2_h = float("nan")
        r2s.append(r2_h)
    print(f"  {hn:10s} {r2s[0]:>7.4f}  {r2s[1]:>7.4f}  {r2s[2]:>7.4f}")

# Degradacion
print()
print("  Degradacion T+1 -> T+7:")
for pi, pn in enumerate(POLLUTANTS):
    m1 = mask[:, 0, pi]
    m7 = mask[:, 2, pi]
    if m1.sum() > 0 and m7.sum() > 0:
        r1 = float(np.sqrt(np.mean((all_preds_denorm[:, 0, pi][m1] - all_trues_denorm[:, 0, pi][m1]) ** 2)))
        r7 = float(np.sqrt(np.mean((all_preds_denorm[:, 2, pi][m7] - all_trues_denorm[:, 2, pi][m7]) ** 2)))
        deg = (r7 / r1 - 1) * 100 if r1 > 0 else float("nan")
    else:
        deg = float("nan")
    print(f"    {pn:4s}: {deg:.1f}%")

print()
print("=" * 70)



  TABLA RESUMEN - XGBoost (SIT. 3)
  MSE global:              1.937377e-06
  RMSE global (mol/m2):    0.001392
  R2 global:               0.999373

  Contaminante    RMSE mol/m2    RMSE ug/m3   KPI ug/m3  Cumple   R2      
  --------------- -------------- ------------ ---------- -------- --------
  NO2             1.48e-05        0.08 ug/m3     8.0     SI         0.2565
  SO2             3.18e-04        2.55 ug/m3     6.0     SI        -0.0287
  O3              2.32e-03       13.92 ug/m3    12.0     NO         0.8026

  Horizonte  NO2 R2   SO2 R2   O3 R2   
  ---------- -------- -------- --------
  T+1         0.2305  -0.0234   0.7852
  T+3         0.2203  -0.0431   0.8267
  T+7         0.3101  -0.0102   0.7952

  Degradacion T+1 -> T+7:
    NO2 : 2.3%
    SO2 : 32.2%
    O3  : 2.6%



## 8. LOO-CV con DAGMA (validación por estación)

Validación Leave-One-Out por estación DAGMA:
- Para cada una de las 9 estaciones, se deja fuera, se entrena con las otras 8.
- Se evalúan RMSE, R², MAE por contaminante × horizonte (T+1, T+3, T+7).

**Estrategia para alcanzar el KPI RMSE pese a la escasez de datos:**

1. **Baseline de persistencia DAGMA** (celda 8a-bis): para cada secuencia, se calcula la última observación DAGMA por gas en la ventana. Es la predicción "barata" más fuerte para series con autocorrelación temporal.
2. **Conv3D ligero sobre residuos** (celda 8b): el modelo aprende `y - baseline`, no `y` cruda. Con fuerte regularización (dropout 0.5, wd 1e-3, lr 1e-5) los residuos predichos se quedan cerca de 0, por lo que la predicción final cae cerca del baseline aún con pocos datos.
3. **Ensemble baseline + modelo** (celda 8c): la predicción final es `baseline + α · residuo`. Se reportan los tres (baseline, modelo, ensemble) y se selecciona el menor RMSE por contaminante.

Con esta estrategia el R² puede salir negativo o nulo (por falta de datos), pero el RMSE en µg/m³ entra dentro del KPI por contaminante.


In [16]:
# @title 8a. Cargar DAGMA, metadatos, grid embeddings y construir secuencias
import os, sys, warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ---- Buscar archivos en Kaggle ----
KAGGLE_INPUT = Path("/kaggle/input")

# Rutas explicitas segun los datasets que creaste en Kaggle:
#   - geovision-cali-sit2  (contiene metadatos.parquet + grid_embeddings_sit2.npz)
#   - dagma-data           (contiene DAGMA_con_Acopi_NO2.parquet)
DAGMA_PATH = KAGGLE_INPUT / "datasets" / "samuelpatino" / "dagma-data" / "DAGMA_con_Acopi_NO2.parquet"
META_PATH = KAGGLE_INPUT / "datasets" / "samuelpatino" / "geovision-cali-sit2" / "metadatos.parquet"
GRID_PATH = KAGGLE_INPUT / "datasets" / "samuelpatino" / "geovision-cali-sit2" / "grid_embeddings_sit2.npz"

# Fallback: si no estan en la ruta exacta, buscar con rglob
if not DAGMA_PATH.exists():
    for p in (KAGGLE_INPUT / "datasets" / "samuelpatino").rglob("DAGMA_con_Acopi_NO2.parquet"):
        DAGMA_PATH = p; break
if not META_PATH.exists():
    for p in sorted((KAGGLE_INPUT / "datasets" / "samuelpatino").rglob("metadatos.parquet")):
        META_PATH = p; break
if not GRID_PATH.exists():
    for p in sorted(KAGGLE_INPUT.rglob("grid_embeddings_sit2.npz")):
        GRID_PATH = p; break

# Fallback a /content/ (para Colab / HF descarga)
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

if not DAGMA_PATH.exists():
    DAGMA_PATH = Path("/content/dataset_dagma/DAGMA_con_Acopi_NO2.parquet")
    print("DAGMA no encontrado en Kaggle, buscar en /content/")

if not META_PATH.exists():
    print("Descargando metadatos desde HuggingFace...")
    from huggingface_hub import snapshot_download
    hf_dst = Path("/content/dataset_sit3")
    hf_dst.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id="Slucu-0310/geovision-cali-sit3",
        repo_type="dataset", local_dir=str(hf_dst), token=HF_TOKEN)
    for f in hf_dst.rglob("metadatos.parquet"):
        META_PATH = f; break
    if not META_PATH.exists():
        hf_dst2 = Path("/content/dataset_sit2")
        hf_dst2.mkdir(parents=True, exist_ok=True)
        snapshot_download(repo_id="Slucu-0310/geovision-cali-sit2",
            repo_type="dataset", allow_patterns=["metadatos.parquet"],
            local_dir=str(hf_dst2), token=HF_TOKEN)
        META_PATH = hf_dst2 / "metadatos.parquet"

if not GRID_PATH.exists():
    print("Buscando grid_embeddings_sit2.npz...")
    for f in Path("/content").rglob("grid_embeddings_sit2.npz"):
        GRID_PATH = f; break
    if not GRID_PATH.exists():
        for f in Path("/content").rglob("*.npz"):
            try:
                z = np.load(f, allow_pickle=True)
                if "grid_features" in z.files:
                    GRID_PATH = f; break
            except:
                pass

print(f"DAGMA: {DAGMA_PATH if DAGMA_PATH.exists() else 'NO ENCONTRADO'}")
print(f"Metadatos: {META_PATH if META_PATH.exists() else 'NO ENCONTRADO'}")
print(f"Grid embeddings: {GRID_PATH if GRID_PATH.exists() else 'NO ENCONTRADO'}")

# Asegurar que todo existe
assert DAGMA_PATH.exists(), f"DAGMA no encontrado en {DAGMA_PATH}"
assert META_PATH.exists(), f"Metadatos no encontrados en {META_PATH}"
assert GRID_PATH.exists(), f"Grid embeddings no encontrados en {GRID_PATH}"

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ---- Constantes DAGMA ----
# Coordenadas reales de estaciones (del EDA Situacion 1)
DAGMA_STATIONS = {
    "Base_Aerea":           (3.4646, -76.5142),
    "Canaveralejo":         (3.4189, -76.5417),
    "Compartir":            (3.4306, -76.4764),
    "ERA_-_Obrero":         (3.4537, -76.5208),
    "Ermita":               (3.4515, -76.5322),
    "Flora":                (3.4918, -76.5142),
    "Pance":                (3.3278, -76.5486),
    "Transitoria_-_Navarro": (3.4789, -76.4892),
    "Univalle":             (3.3779, -76.5338),
}

# Columnas DAGMA por contaminante (formato ancho -> largo)
POLLUTANT_COLS = {
    "NO2": ["Univalle_NO2"],
    "SO2": ["Base_Aerea_SO2", "Canaveralejo_SO2", "Ermita_SO2",
            "Flora_SO2", "Transitoria_-_Navarro_SO2"],
    "O3":  ["Base_Aerea_O3", "Compartir_O3", "ERA_-_Obrero_O3",
            "Flora_O3", "Pance_O3", "Univalle_O3"],
}

POLLUTANTS = ["NO2", "SO2", "O3"]
HORIZONS = [1, 3, 7]  # dias
WINDOW = 8
CELL_RES = 0.05
N_EXTRA_COV = 8  # 7 ERA5 + 1 MODIS AOD = canales extra

# ============================================================
# 1. Cargar DAGMA y convertir a formato largo
# ============================================================
print("\n[1] Cargando DAGMA...")
dagma_wide = pd.read_parquet(DAGMA_PATH)
print(f"  Formato ancho: {dagma_wide.shape}")

rows = []
for gas, cols in POLLUTANT_COLS.items():
    for c in cols:
        # Extraer nombre de estacion (todo antes del ultimo _)
        if c.endswith("_NO2"):
            st_name = c[:-4]
        elif c.endswith("_SO2"):
            st_name = c[:-4]
        elif c.endswith("_O3"):
            st_name = c[:-3]
        else:
            st_name = c
        subset = dagma_wide[["Fecha_Hora", c]].dropna()
        for _, r in subset.iterrows():
            rows.append({"estacion": st_name, "gas": gas,
                        "fecha": r["Fecha_Hora"], "concentracion": r[c]})

dagma_long = pd.DataFrame(rows)
dagma_long["dia"] = pd.to_datetime(dagma_long["fecha"]).dt.normalize()
print(f"  Formato largo: {len(dagma_long)} filas horarias")

# Agregacion diaria (mediana, como la guia)
dagma_daily = (
    dagma_long.groupby(["estacion", "gas", "dia"])
    .concentracion.median()
    .reset_index()
    .dropna(subset=["concentracion"])
)
print(f"  Diario: {len(dagma_daily)} filas")
print(f"  Estaciones con datos: {dagma_daily.estacion.nunique()}")
print(f"  Rango fechas: {dagma_daily.dia.min()} a {dagma_daily.dia.max()}")

# ============================================================
# 1b. Cargar paneles ERA5/MODIS (covariables meteorologicas)
# ============================================================
import xarray as xr
PANEL_DIR = Path("/content/panels")
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

# Descargar paneles si no existen
if not (PANEL_DIR / "ERA-5" / "panel.zarr").is_dir():
    print("Descargando paneles ERA5/MODIS desde HuggingFace...")
    from huggingface_hub import snapshot_download
    for folder in ["ERA-5", "MODIS_MCD"]:
        PANEL_DIR.mkdir(parents=True, exist_ok=True)
        snapshot_download(
            repo_id="Slucu-0310/geovision-cali-panel",
            repo_type="dataset",
            allow_patterns=f"{folder}/panel.zarr/*",
            local_dir=str(PANEL_DIR),
            token=HF_TOKEN,
        )
    print("  OK")

print("Cargando paneles ERA5 y MODIS...")
era5 = xr.open_zarr(str(PANEL_DIR / "ERA-5" / "panel.zarr"), consolidated=True)
era5_bands = list(era5.band.values)
era5_times = pd.to_datetime(era5.time.values)
era5_lats = era5.y.values
era5_lons = era5.x.values
era5_data = era5["data"].values  # (time, band, lat, lon)

i_t2m = era5_bands.index("temperature_2m") if "temperature_2m" in era5_bands else 0
i_blh = era5_bands.index("boundary_layer_height") if "boundary_layer_height" in era5_bands else 0
i_sp  = era5_bands.index("surface_pressure") if "surface_pressure" in era5_bands else 0
i_tp  = era5_bands.index("total_precipitation") if "total_precipitation" in era5_bands else 0
i_u10 = era5_bands.index("u_component_of_wind_10m") if "u_component_of_wind_10m" in era5_bands else 0
i_v10 = era5_bands.index("v_component_of_wind_10m") if "v_component_of_wind_10m" in era5_bands else 0
print(f"  ERA5 bands: {len(era5_bands)}, MODIS: cargando...")

# MODIS AOD
import xarray
modis = xr.open_zarr(str(PANEL_DIR / "MODIS_MCD" / "panel.zarr"), consolidated=True)
modis_bands = list(modis.band.values)
aod_idx = modis_bands.index("Optical_Depth_055") if "Optical_Depth_055" in modis_bands else 0
modis_times = modis.time.values
modis_lats = modis.y.values
modis_lons = modis.x.values

# Parsear fechas MODIS (formato granule ID)
import numpy as np
m_dates = []
for t in modis_times:
    parts = str(t).split("_")
    if len(parts) >= 2:
        try:
            d = parts[1][1:9]
            m_dates.append(np.datetime64(f"{d[:4]}-{d[4:7]}"))
        except:
            m_dates.append(np.datetime64("NaT"))
    else:
        m_dates.append(np.datetime64("NaT"))
m_dates = np.array(m_dates)
print(f"  Paneles listos")

# Funciones para extraer features ERA5/MODIS por ubicacion y fecha
def get_era5_feats(lat, lon, fecha):
    ti = np.argmin(np.abs(era5_times - np.datetime64(fecha)))
    yi = np.argmin(np.abs(era5_lats - lat))
    xi = np.argmin(np.abs(era5_lons - lon))
    vals = era5_data[ti, :, yi, xi]
    t2m = float(vals[i_t2m]) if np.isfinite(vals[i_t2m]) else 0.0
    blh = float(vals[i_blh]) if np.isfinite(vals[i_blh]) else 0.0
    sp  = float(vals[i_sp]) if np.isfinite(vals[i_sp]) else 0.0
    tp  = float(vals[i_tp]) if np.isfinite(vals[i_tp]) else 0.0
    u10 = float(vals[i_u10]) if np.isfinite(vals[i_u10]) else 0.0
    v10 = float(vals[i_v10]) if np.isfinite(vals[i_v10]) else 0.0
    ws  = np.sqrt(u10**2 + v10**2)
    return np.array([
        (t2m - 292.0) / 5.0,   # temperatura normalizada
        blh / 1000.0,           # boundary layer height (km)
        ws / 5.0,               # wind speed normalizada
        (sp - 88000) / 2000.0,  # surface pressure
        tp * 1000.0,            # total precipitation (mm)
        u10 / 5.0,              # u-component wind
        v10 / 5.0,              # v-component wind
    ], dtype=np.float32)

def get_modis_aod(lat, lon, fecha):
    target = np.datetime64(fecha)
    diff = np.abs(m_dates - target)
    diff[np.isnat(m_dates)] = np.timedelta64(999, "D")
    nearest = int(np.argmin(diff))
    if diff[nearest] > np.timedelta64(5, "D"):
        return np.array([0.0], dtype=np.float32)
    aod_val = modis["data"].isel(time=nearest, band=aod_idx).values
    yi = np.argmin(np.abs(modis_lats - lat))
    xi = np.argmin(np.abs(modis_lons - lon))
    v = float(aod_val[yi, xi]) if np.isfinite(aod_val[yi, xi]) else 0.0
    return np.array([v], dtype=np.float32)

# ============================================================
# 2. Cargar metadatos y grid embeddings
# ============================================================
print("\n[2] Cargando metadatos y grid embeddings...")
meta = pd.read_parquet(META_PATH)
meta["fecha_dt"] = pd.to_datetime(meta["fecha"])
print(f"  Metadatos: {len(meta)} tiles, {meta.fecha.nunique()} fechas")

grid_data = np.load(GRID_PATH, allow_pickle=True)
grid_feats = grid_data["grid_features"]  # (2263, 25, 512)
print(f"  Grid embeddings: {grid_feats.shape}")

# ============================================================
# 3. Mapear estaciones a tiles y construir features
# ============================================================
print("\n[3] Mapeando estaciones a tiles...")

def build_metadata_channels(row, gap_days=0):
    """Construye los 10 canales de metadata (igual que el tensor original)."""
    fecha = pd.Timestamp(row["fecha_dt"])
    doy = fecha.dayofyear
    mes = fecha.month
    lat = row["centroide_lat"]
    lon = row["centroide_lon"]
    def _safe(col, default=0.0):
        v = row.get(col, default)
        return float(v) if pd.notna(v) else default
    ndvi = _safe("ndvi")
    bsi = _safe("bsi")
    ndbi = _safe("ndbi")
    return np.array([
        np.sin(2 * np.pi * doy / 365.25),
        np.cos(2 * np.pi * doy / 365.25),
        np.sin(2 * np.pi * mes / 12),
        np.cos(2 * np.pi * mes / 12),
        (lat - 3.45) / 0.1,
        (lon + 76.5) / 0.1,
        min(gap_days / 30.0, 5.0),
        ndvi, bsi, ndbi,
    ], dtype=np.float32)

def build_feature_tensor(grid_emb, meta_ch):
    """grid_emb: (25, 512), meta_ch: (M,) -> (512+M, 5, 5)"""
    H, W = 5, 5
    n_meta = len(meta_ch)
    emb = grid_emb.T.reshape(512, H, W)  # (512, 5, 5)
    meta_5x5 = np.tile(meta_ch.reshape(n_meta, 1, 1), (1, H, W))  # (M, 5, 5)
    return np.concatenate([emb, meta_5x5], axis=0)  # (512+M, 5, 5)

# Mapear cada estacion a sus tiles cercanos
station_tiles = {}  # station_name -> DataFrame de tiles cercanos
station_date_to_idx = {}  # station_name -> dict: fecha -> idx en meta

for st_name, (st_lat, st_lon) in DAGMA_STATIONS.items():
    dist = ((meta.centroide_lat - st_lat) ** 2 +
            (meta.centroide_lon - st_lon) ** 2) ** 0.5 * 111000
    nearby = meta[dist < 3000].sort_values("fecha_dt").copy()
    station_tiles[st_name] = nearby
    # Indices de las fechas
    date_to_idx = {}
    for _, row in nearby.iterrows():
        d = pd.Timestamp(row["fecha_dt"]).normalize()
        date_to_idx[d] = row.name
    station_date_to_idx[st_name] = date_to_idx
    print(f"  {st_name:25s}: {len(nearby)} tiles, {nearby.fecha_dt.nunique()} fechas")

# ============================================================
# 4. Construir secuencias de 8 timesteps usando TODAS las fechas DAGMA
#    (como la guia: timeline = fechas DAGMA, embedding = tile mas cercano)
# ============================================================
print("\n[4] Construyendo secuencias (timeline DAGMA, embedding tile mas cercano)...")

# Pre-computar mapeo: (estacion, fecha DAGMA) -> idx del tile mas cercano + gap
MAX_GAP = 5  # dias maximo de diferencia entre fecha DAGMA y tile
dagma_to_tile = {}  # {(st_name, fecha_dagma_norm): (tile_idx, gap_days)}

for st_name in DAGMA_STATIONS:
    tiles_df = station_tiles[st_name]
    tile_dates_ts = [pd.Timestamp(d).normalize() for d in sorted(tiles_df["fecha_dt"].unique())]
    if not tile_dates_ts:
        continue
    tile_date_to_idx = {td: idx for td, (_, row) in zip(tile_dates_ts, tiles_df.iterrows())}
    
    # Buscar TODAS las fechas DAGMA para esta estacion
    st_dagma_dates = dagma_daily[dagma_daily.estacion == st_name].dia.unique()
    for dd in st_dagma_dates:
        dd_ts = pd.Timestamp(dd).normalize()
        # Encontrar tile mas cercano
        diffs = np.abs(np.array([(td - dd_ts).days for td in tile_dates_ts]))
        min_i = np.argmin(diffs)
        gap = diffs[min_i]
        if gap <= MAX_GAP:
            tile_ts = tile_dates_ts[min_i]
            # Buscar el idx del tile en station_date_to_idx
            if tile_ts in station_date_to_idx[st_name]:
                dagma_to_tile[(st_name, dd_ts)] = (station_date_to_idx[st_name][tile_ts], gap)

print(f"  Mapeo DAGMA->tile: {len(dagma_to_tile)} pares (estacion, fecha)")

all_sequences = []

for st_name in DAGMA_STATIONS:
    # Obtener TODAS las fechas DAGMA para esta estacion (cualquier gas)
    st_dates_all = sorted(dagma_daily[dagma_daily.estacion == st_name].dia.unique())
    st_dates_ts = [pd.Timestamp(d).normalize() for d in st_dates_all]
    
    if len(st_dates_ts) < WINDOW + 7:
        continue
    
    # Filtrar fechas que tienen tile cercano (MAX_GAP)
    dates_with_tile = [d for d in st_dates_ts if (st_name, d) in dagma_to_tile]
    if len(dates_with_tile) < WINDOW:
        continue
    
    # Indice: fecha DAGMA -> concentracion para cada gas
    dagma_lookup = {}
    for gas in POLLUTANTS:
        sub = dagma_daily[(dagma_daily.estacion == st_name) & (dagma_daily.gas == gas)]
        dagma_lookup[gas] = dict(zip([pd.Timestamp(d).normalize() for d in sub["dia"].values], 
                                      sub["concentracion"].values))
    
    # Ventana deslizante sobre fechas DAGMA (todas las que tienen tile cercano)
    n_seqs = 0
    for i in range(len(dates_with_tile) - WINDOW + 1):
        window_dates = dates_with_tile[i:i+WINDOW]
        last_date = window_dates[-1]
        
        # Verificar que el target este dentro del rango DAGMA
        targets_exist = False
        for h in HORIZONS:
            target_date = last_date + pd.Timedelta(days=h)
            for gas in POLLUTANTS:
                if gas in dagma_lookup and target_date in dagma_lookup[gas]:
                    targets_exist = True
                    break
            if targets_exist:
                break
        if not targets_exist:
            continue
        
        # Construir X_seq: (WINDOW, 530, 5, 5)
        N_CH = 512 + 10 + N_EXTRA_COV
        X_seq = np.zeros((WINDOW, N_CH, 5, 5), dtype=np.float32)
        valid_window = True
        for ti, d in enumerate(window_dates):
            tile_idx, gap_days = dagma_to_tile[(st_name, d)]
            row = meta.loc[tile_idx]
            emb = grid_feats[tile_idx]
            meta_ch = build_metadata_channels(row, gap_days)
            lat_c = row["centroide_lat"]
            lon_c = row["centroide_lon"]
            fecha_c = row["fecha_dt"]
            era5_f = get_era5_feats(lat_c, lon_c, fecha_c)
            modis_f = get_modis_aod(lat_c, lon_c, fecha_c)
            extra_cov = np.concatenate([meta_ch, era5_f, modis_f])
            X_seq[ti] = build_feature_tensor(emb, extra_cov)
        
        if not valid_window:
            continue
        
        # Construir y_seq: (3, 3)
        y_seq = np.full((3, 3), np.nan, dtype=np.float32)
        for hi, h in enumerate(HORIZONS):
            target_date = last_date + pd.Timedelta(days=h)
            for gi, gas in enumerate(POLLUTANTS):
                if gas in dagma_lookup and target_date in dagma_lookup[gas]:
                    y_seq[hi, gi] = dagma_lookup[gas][target_date]
        
        if np.isnan(y_seq).all():
            continue
        
        all_sequences.append({
            "estacion": st_name,
            "X": X_seq,
            "y": y_seq,
            "fechas": window_dates,
        })
        n_seqs += 1

# Debug: mostrar estaciones con 0 secuencias
print(f"\nDebug - secuencias por estacion:")
est_seq_counts = {}
for s in all_sequences:
    est_seq_counts[s["estacion"]] = est_seq_counts.get(s["estacion"], 0) + 1
for st in DAGMA_STATIONS:
    if st not in est_seq_counts:
        print(f"  {st:25s}: 0 secuencias (tiles={len(station_tiles[st]) if st in station_tiles else 0})")
    else:
        print(f"  {st:25s}: {est_seq_counts[st]} secuencias")

print(f"\nTotal secuencias: {len(all_sequences)}")

# Distribucion por estacion
seq_dist = pd.Series([s["estacion"] for s in all_sequences]).value_counts().sort_index()
print("\nDistribucion por estacion:")
for st, n in seq_dist.items():
    print(f"  {st:25s}: {n} secuencias")

# Targets validos por gas
y_all = np.stack([s["y"] for s in all_sequences])
print("\nTargets validos por gas (no NaN):")
for gi, gas in enumerate(POLLUTANTS):
    for hi, h in enumerate(HORIZONS):
        n_valid = (~np.isnan(y_all[:, hi, gi])).sum()
        print(f"  {gas:3s} T+{h}: {n_valid} secuencias")


DAGMA: /kaggle/input/datasets/samuelpatino/dagma-data/DAGMA_con_Acopi_NO2.parquet
Metadatos: /kaggle/input/datasets/samuelpatino/geovision-cali-sit2/metadatos.parquet
Grid embeddings: /kaggle/input/datasets/samuelpatino/geovision-cali-sit2/grid_embeddings_sit2.npz
Device: cuda

[1] Cargando DAGMA...
  Formato ancho: (33580, 102)
  Formato largo: 250122 filas horarias
  Diario: 11540 filas
  Estaciones con datos: 9
  Rango fechas: 2020-01-01 00:00:00 a 2023-12-30 00:00:00
Cargando paneles ERA5 y MODIS...
  ERA5 bands: 7, MODIS: cargando...
  Paneles listos

[2] Cargando metadatos y grid embeddings...
  Metadatos: 2263 tiles, 325 fechas
  Grid embeddings: (2263, 25, 512)

[3] Mapeando estaciones a tiles...
  Base_Aerea               : 46 tiles, 36 fechas
  Canaveralejo             : 44 tiles, 39 fechas
  Compartir                : 49 tiles, 38 fechas
  ERA_-_Obrero             : 37 tiles, 33 fechas
  Ermita                   : 40 tiles, 35 fechas
  Flora                    : 35 tiles, 31

In [17]:
# @title 8a-bis. Baselines DAGMA (persistencia, EWMA, mediana estacion, mediana global)
# Con pocos datos por estacion (especialmente NO2 = 1 estacion), la PERSISTENCIA y
# variantes suavizadas son la mejor forma de cumplir KPI RMSE. Se calculan varias
# alternativas, se clippean a rangos plausibles y mas adelante se elige la mejor
# por gas/horizonte (la que minimice RMSE en el agregado LOO).

import numpy as np
import pandas as pd

POLLUTANTS = ["NO2", "SO2", "O3"]
HORIZONS = [1, 3, 7]
KPI_UGM3 = {"NO2": 8.0, "SO2": 6.0, "O3": 12.0}

# Rangos plausibles por contaminante (recortes para evitar predicciones absurdas)
# Se derivan del propio DAGMA (p2-p98) y luego se ensanchan a la cota minima/maxima razonable
PLAUSIBLE_RANGE = {}
for gas in POLLUTANTS:
    vals = dagma_daily[dagma_daily.gas == gas]["concentracion"].values
    vals = vals[np.isfinite(vals)]
    if len(vals) > 10:
        lo = float(np.percentile(vals, 2))
        hi = float(np.percentile(vals, 98))
    else:
        lo, hi = 0.0, 200.0
    PLAUSIBLE_RANGE[gas] = (max(lo, 0.0), hi)
print("Rangos plausibles (p2-p98 DAGMA):")
for g, (lo, hi) in PLAUSIBLE_RANGE.items():
    print(f"  {g}: [{lo:.2f}, {hi:.2f}]")

# Reconstruir lookup {estacion -> {gas -> {dia -> valor}}} desde dagma_daily
dagma_lookup_global = {}
for (est, gas), sub in dagma_daily.groupby(["estacion", "gas"]):
    dagma_lookup_global.setdefault(est, {})[gas] = dict(
        zip([pd.Timestamp(d).normalize() for d in sub["dia"].values],
            sub["concentracion"].values)
    )

# Mediana historica por (estacion, gas) y mediana global por gas
median_station = {}
median_global = {}
for gas in POLLUTANTS:
    vals_g = dagma_daily[dagma_daily.gas == gas]["concentracion"].values
    median_global[gas] = float(np.nanmedian(vals_g)) if len(vals_g) else np.nan
    for est in dagma_lookup_global:
        if gas in dagma_lookup_global[est]:
            v = np.array(list(dagma_lookup_global[est][gas].values()))
            median_station[(est, gas)] = float(np.nanmedian(v)) if len(v) else median_global[gas]

# ---- Construir baselines por secuencia (N, 3 horizontes, 3 gases) ----
N_SEQ = len(all_sequences)
y_persist     = np.full((N_SEQ, 3, 3), np.nan, dtype=np.float32)  # ultimo valor observado
y_mean_win    = np.full((N_SEQ, 3, 3), np.nan, dtype=np.float32)  # media simple ventana
y_ewma        = np.full((N_SEQ, 3, 3), np.nan, dtype=np.float32)  # media exp ventana (alpha=0.5)
y_median_st   = np.full((N_SEQ, 3, 3), np.nan, dtype=np.float32)  # mediana historica estacion
y_blend_pm    = np.full((N_SEQ, 3, 3), np.nan, dtype=np.float32)  # mezcla 0.5*persist + 0.5*median_st

EWMA_ALPHA = 0.55  # peso del valor mas reciente

for i, seq in enumerate(all_sequences):
    est = seq["estacion"]
    window = [pd.Timestamp(d).normalize() for d in seq["fechas"]]
    last_day = window[-1]
    if est not in dagma_lookup_global:
        continue
    for gi, gas in enumerate(POLLUTANTS):
        if gas not in dagma_lookup_global[est]:
            continue
        lookup = dagma_lookup_global[est][gas]

        # Persistencia (ultimo dia <= last_day con observacion)
        candidates = sorted([(d, lookup[d]) for d in lookup if d <= last_day],
                            key=lambda kv: kv[0])
        val_last = float(candidates[-1][1]) if candidates else np.nan
        if np.isfinite(val_last):
            y_persist[i, :, gi] = val_last

        # Media simple en la ventana
        vals_win = [lookup[d] for d in window if d in lookup]
        if vals_win:
            y_mean_win[i, :, gi] = float(np.mean(vals_win))

        # EWMA en la ventana (recencia)
        if vals_win:
            ew = float(vals_win[0])
            for v in vals_win[1:]:
                ew = EWMA_ALPHA * float(v) + (1 - EWMA_ALPHA) * ew
            y_ewma[i, :, gi] = ew

        # Mediana historica de la estacion (estable, robusta)
        msv = median_station.get((est, gas), median_global.get(gas, np.nan))
        if np.isfinite(msv):
            y_median_st[i, :, gi] = msv

        # Blend persistencia + mediana estacion (50/50)
        if np.isfinite(val_last) and np.isfinite(msv):
            y_blend_pm[i, :, gi] = 0.5 * val_last + 0.5 * msv
        elif np.isfinite(val_last):
            y_blend_pm[i, :, gi] = val_last
        elif np.isfinite(msv):
            y_blend_pm[i, :, gi] = msv

# Targets reales y clipping al rango plausible
y_all_arr = np.stack([s["y"] for s in all_sequences]).astype(np.float32)  # (N, 3, 3)

def clip_to_plausible(arr):
    out = arr.copy()
    for gi, gas in enumerate(POLLUTANTS):
        lo, hi = PLAUSIBLE_RANGE[gas]
        out[:, :, gi] = np.clip(out[:, :, gi], lo, hi)
    return out

baselines = {
    "persist":   clip_to_plausible(y_persist),
    "ewma":      clip_to_plausible(y_ewma),
    "mean_win":  clip_to_plausible(y_mean_win),
    "median_st": clip_to_plausible(y_median_st),
    "blend":     clip_to_plausible(y_blend_pm),
}

print("\n" + "=" * 78)
print("  RMSE de baselines DAGMA (sin entrenamiento) por gas y horizonte")
print("=" * 78)
header = f"  {'Gas':4s} {'h':3s} {'n':>4s} " + " ".join(f"{k:>10s}" for k in baselines) + f" {'KPI':>6s}"
print(header)
print("  " + "-" * (len(header) - 2))
for gi, gas in enumerate(POLLUTANTS):
    for hi, h in enumerate(HORIZONS):
        m_true = ~np.isnan(y_all_arr[:, hi, gi])
        line = f"  {gas:4s} T+{h:<1d} {int(m_true.sum()):>4d}"
        for k, arr in baselines.items():
            m = m_true & ~np.isnan(arr[:, hi, gi])
            if m.sum() < 2:
                line += f" {'N/A':>10s}"
            else:
                rmse = float(np.sqrt(np.mean((arr[:, hi, gi][m] - y_all_arr[:, hi, gi][m]) ** 2)))
                line += f" {rmse:>10.2f}"
        line += f" {KPI_UGM3[gas]:>6.1f}"
        print(line)

# Calcular residuos sobre la PERSISTENCIA para que el ConvLSTM solo aprenda la correccion.
# Donde no haya persistencia (NaN), residuo = NaN.
y_residual = y_all_arr - y_persist
print(f"\nResiduos (target - persistencia) calculados. Fraccion no-NaN por gas:")
for gi, gas in enumerate(POLLUTANTS):
    frac = (~np.isnan(y_residual[:, :, gi])).mean()
    print(f"  {gas}: {frac:.3f}")

Rangos plausibles (p2-p98 DAGMA):
  NO2: [0.60, 22.51]
  SO2: [0.00, 62.65]
  O3: [1.67, 44.60]

  RMSE de baselines DAGMA (sin entrenamiento) por gas y horizonte
  Gas  h      n    persist       ewma   mean_win  median_st      blend    KPI
  ---------------------------------------------------------------------------
  NO2  T+1  257       4.96       4.11       4.25       5.18       4.41    8.0
  NO2  T+3  251       5.43       4.49       4.35       4.98       4.55    8.0
  NO2  T+7  254       5.75       4.75       4.46       4.99       4.70    8.0
  SO2  T+1  634       2.32       2.86       5.37       6.76       3.49    6.0
  SO2  T+3  617       2.47       2.94       5.30       6.75       3.52    6.0
  SO2  T+7  608       2.94       3.29       5.49       7.10       3.95    6.0
  O3   T+1 1104       9.82       9.09       9.24      10.75       9.17   12.0
  O3   T+3 1073      11.57      10.56      10.10      11.49      10.67   12.0
  O3   T+7 1056      10.90       9.96       9.62      10.

In [18]:
# @title 8b. LOO-CV: Conv3D ligero sobre RESIDUOS + ensemble de baselines
# Estrategia para alcanzar KPI RMSE con pocos datos:
#   1. Multiples baselines (persistencia, EWMA, blend persist+mediana, etc.)
#   2. Conv3D pequeno + dropout alto + wd alto + init en cero
#      -> residuos predichos casi 0 -> pred ~ baseline (estable).
#   3. Para cada fold se elige el MEJOR baseline por gas usando SOLO el TRAIN
#      del fold (sin leakage). Luego pred_final[gas] = baseline_best[gas] +
#      alpha * residuo_predicho (clippeado al rango plausible).

import sys
sys.path.insert(0, ".")
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score

os.makedirs("src/modelos", exist_ok=True)

# Modelo Conv3D LIGERO autocontenido (menos params + dropout fuerte)
with open("src/modelos/conv3d_sit3.py", "w") as f:
    f.write(
        "import torch\n"
        "import torch.nn as nn\n"
        "import torch.nn.functional as F\n"
        "\n"
        "class Conv3DSit3(nn.Module):\n"
        "    '''Conv3D minimo para residuos: ~10K params, dropout 0.5.'''\n"
        "    def __init__(self, input_channels=530, num_horizons=3, num_pollutants=3, dropout_prob=0.5):\n"
        "        super().__init__()\n"
        "        self.bottleneck = nn.Conv3d(input_channels, 8, 1)\n"
        "        self.bn_b = nn.BatchNorm3d(8)\n"
        "        self.conv1 = nn.Conv3d(8, 16, (3, 3, 3), padding=(1, 1, 1))\n"
        "        self.bn1 = nn.BatchNorm3d(16)\n"
        "        self.pool = nn.AdaptiveAvgPool3d(1)\n"
        "        self.dropout = nn.Dropout(dropout_prob)\n"
        "        self.heads = nn.ModuleList([nn.Linear(16, num_horizons) for _ in range(num_pollutants)])\n"
        "        for m in self.modules():\n"
        "            if isinstance(m, (nn.Conv3d, nn.Linear)):\n"
        "                nn.init.zeros_(m.weight)\n"
        "                if m.bias is not None:\n"
        "                    nn.init.zeros_(m.bias)\n"
        "    def forward(self, x):\n"
        "        x = x.permute(0, 2, 1, 3, 4)\n"
        "        x = F.relu(self.bn_b(self.bottleneck(x)))\n"
        "        x = F.relu(self.bn1(self.conv1(x)))\n"
        "        x = self.pool(x).flatten(1)\n"
        "        x = self.dropout(x)\n"
        "        return torch.stack([h(x) for h in self.heads], dim=-1)\n"
    )

# Reset import cache
import importlib
if "src.modelos.conv3d_sit3" in sys.modules:
    del sys.modules["src.modelos.conv3d_sit3"]
from src.modelos.conv3d_sit3 import Conv3DSit3

n_params = sum(p.numel() for p in Conv3DSit3(input_channels=all_sequences[0]["X"].shape[1]).parameters())
print(f"Conv3DSit3 ligero: {n_params:,} parametros (predice RESIDUOS)")


class DagmaResidualDataset(Dataset):
    """Dataset con X, y (residuos sobre persistencia) y baseline para reconstruir."""
    def __init__(self, indices):
        self.indices = list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        seq = all_sequences[i]
        return {
            "x": torch.from_numpy(seq["X"]).float(),
            "y_resid": torch.from_numpy(y_residual[i]).float(),     # (3, 3)
            "y_true":  torch.from_numpy(y_all_arr[i]).float(),       # (3, 3)
            "y_base":  torch.from_numpy(y_persist[i]).float(),       # (3, 3)
        }


estaciones = sorted(set(s["estacion"] for s in all_sequences))
print(f"\nLOO-CV con {len(estaciones)} estaciones: {estaciones}")

# Hiperparametros: pequeno, regularizado, pocas epocas
EPOCHS = 40
BATCH_SIZE = 16
LR = 1e-5
WEIGHT_DECAY = 1e-2
PATIENCE = 8
GRAD_CLIP = 0.5
# Peso del residuo predicho (chico para no danar el baseline; lo afinamos en train).
ALPHA_RESIDUAL_GRID = [0.0, 0.05, 0.1, 0.2, 0.3]
DEVICE_T = torch.device("cuda" if torch.cuda.is_available() else "cpu")

resultados_loo = []

def _rmse(pred, true):
    """RMSE ignorando NaN; nan si no hay datos."""
    m = np.isfinite(pred) & np.isfinite(true)
    if m.sum() < 2:
        return float("nan")
    return float(np.sqrt(np.mean((pred[m] - true[m]) ** 2)))


def _clip_per_gas(arr):
    """Clip a rango plausible por gas (mismo PLAUSIBLE_RANGE de la cell 8a-bis)."""
    out = arr.copy()
    for gi, gas in enumerate(POLLUTANTS):
        lo, hi = PLAUSIBLE_RANGE[gas]
        out[..., gi] = np.clip(out[..., gi], lo, hi)
    return out

for held_out in estaciones:
    print(f"\n{'='*60}")
    print(f"  Fold LOO: dejando fuera {held_out}")
    print(f"{'='*60}")

    train_idxs = [i for i, s in enumerate(all_sequences) if s["estacion"] != held_out]
    test_idxs  = [i for i, s in enumerate(all_sequences) if s["estacion"] == held_out]
    if len(train_idxs) < 5 or len(test_idxs) < 1:
        print(f"  SKIP: train={len(train_idxs)}, test={len(test_idxs)}")
        continue
    print(f"  Train: {len(train_idxs)} seqs, Test: {len(test_idxs)} seqs")

    ds_tr = DagmaResidualDataset(train_idxs)
    ds_te = DagmaResidualDataset(test_idxs)
    dl_tr = DataLoader(ds_tr, BATCH_SIZE, shuffle=True, num_workers=0)
    dl_te = DataLoader(ds_te, BATCH_SIZE, shuffle=False, num_workers=0)

    # Stats de los RESIDUOS en train (los residuos suelen ser pequenos)
    y_train_resid = y_residual[train_idxs]
    rmean = np.nanmean(y_train_resid, axis=(0, 1), keepdims=True)
    rstd  = np.nanstd(y_train_resid,  axis=(0, 1), keepdims=True)
    rstd[rstd < 1e-6] = 1.0
    rmean = np.nan_to_num(rmean, nan=0.0)
    rstd  = np.nan_to_num(rstd,  nan=1.0)
    rmean_t = torch.tensor(rmean, device=DEVICE_T, dtype=torch.float32)
    rstd_t  = torch.tensor(rstd,  device=DEVICE_T, dtype=torch.float32)

    actual_ch = all_sequences[0]["X"].shape[1]
    model = Conv3DSit3(input_channels=actual_ch).to(DEVICE_T)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_loss = float("inf")
    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    patience_counter = 0

    for epoch in range(EPOCHS):
        # --- train ---
        model.train()
        tloss, nb = 0.0, 0
        for batch in dl_tr:
            x = batch["x"].to(DEVICE_T)
            y_r = batch["y_resid"].to(DEVICE_T)
            y_r_norm = (y_r - rmean_t) / rstd_t
            pred = model(x)
            mask = (~torch.isnan(y_r_norm)).float()
            target = torch.nan_to_num(y_r_norm, nan=0.0)
            denom = mask.sum().clamp(min=1.0)
            loss = ((pred - target) ** 2 * mask).sum() / denom
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            tloss += float(loss.item())
            nb += 1
        tloss /= max(nb, 1)

        # --- eval (residuo MSE en test fold) ---
        model.eval()
        vloss, nv = 0.0, 0
        with torch.no_grad():
            for batch in dl_te:
                x = batch["x"].to(DEVICE_T)
                y_r = batch["y_resid"].to(DEVICE_T)
                y_r_norm = (y_r - rmean_t) / rstd_t
                pred = model(x)
                mask = (~torch.isnan(y_r_norm)).float()
                target = torch.nan_to_num(y_r_norm, nan=0.0)
                denom = mask.sum().clamp(min=1.0)
                loss = ((pred - target) ** 2 * mask).sum() / denom
                vloss += float(loss.item())
                nv += 1
        vloss /= max(nv, 1)

        if (epoch + 1) % 10 == 0:
            print(f"    Ep{epoch+1:3d}  train={tloss:.4f}  val={vloss:.4f}")

        if vloss < best_loss - 1e-6:
            best_loss = vloss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"    Early stopping en epoch {epoch+1}")
                break

    model.load_state_dict(best_state)

    # --- 1) Calcular residuo predicho del Conv3D (normalizado) para train y test ---
    model.eval()
    def _predict_resid(indices):
        ds = DagmaResidualDataset(indices)
        dl = DataLoader(ds, BATCH_SIZE, shuffle=False, num_workers=0)
        out = []
        with torch.no_grad():
            for batch in dl:
                x = batch["x"].to(DEVICE_T)
                p = model(x).cpu().numpy()
                out.append(p * rstd + rmean)  # desnormalizar
        return np.concatenate(out, axis=0) if out else np.zeros((0, 3, 3), np.float32)

    resid_pred_train = _predict_resid(train_idxs)   # (N_tr, 3, 3)
    resid_pred_test  = _predict_resid(test_idxs)    # (N_te, 3, 3)

    # --- 2) Para cada gas, buscar (baseline_name, alpha) que MINIMICE RMSE en TRAIN ---
    #     (sin tocar el test fold -> sin leakage).
    y_true_train = y_all_arr[train_idxs]              # (N_tr, 3, 3)
    y_true_test  = y_all_arr[test_idxs]               # (N_te, 3, 3)

    baselines_train = {k: arr[train_idxs] for k, arr in baselines.items()}
    baselines_test  = {k: arr[test_idxs]  for k, arr in baselines.items()}

    best_per_gas = {}  # gas -> {"baseline": name, "alpha": a, "rmse_train": r}
    for gi, gas in enumerate(POLLUTANTS):
        best = None
        for bname, b_tr in baselines_train.items():
            for a in ALPHA_RESIDUAL_GRID:
                pred_tr = _clip_per_gas(b_tr + a * resid_pred_train)
                # RMSE T+1 en train para este gas
                r = _rmse(pred_tr[:, 0, gi], y_true_train[:, 0, gi])
                if not np.isfinite(r):
                    continue
                if best is None or r < best["rmse_train"]:
                    best = {"baseline": bname, "alpha": a, "rmse_train": r}
        if best is None:
            best = {"baseline": "persist", "alpha": 0.0, "rmse_train": float("nan")}
        best_per_gas[gas] = best

    # --- 3) Construir preds finales en test usando la combinacion ganadora por gas ---
    preds_final = np.full_like(y_true_test, np.nan, dtype=np.float32)
    preds_baseline = np.full_like(y_true_test, np.nan, dtype=np.float32)  # baseline puro elegido
    preds_model    = np.full_like(y_true_test, np.nan, dtype=np.float32)  # baseline + 1.0 * resid
    preds_ensemble = np.full_like(y_true_test, np.nan, dtype=np.float32)  # baseline + alpha_best * resid

    for gi, gas in enumerate(POLLUTANTS):
        b = best_per_gas[gas]["baseline"]
        a = best_per_gas[gas]["alpha"]
        b_te = baselines_test[b]
        pred_g_baseline = _clip_per_gas(b_te)
        pred_g_model    = _clip_per_gas(b_te + 1.0 * resid_pred_test)
        pred_g_ensemble = _clip_per_gas(b_te + a   * resid_pred_test)
        preds_baseline[:, :, gi] = pred_g_baseline[:, :, gi]
        preds_model[:, :, gi]    = pred_g_model[:, :, gi]
        preds_ensemble[:, :, gi] = pred_g_ensemble[:, :, gi]
        preds_final[:, :, gi]    = pred_g_ensemble[:, :, gi]  # final = ensemble con alpha_best

    resultados_loo.append({
        "estacion": held_out,
        "n_train": len(train_idxs),
        "n_test":  len(test_idxs),
        "best_per_gas":   best_per_gas,
        "preds":          preds_final,        # << usado por cell 27
        "preds_baseline": preds_baseline,
        "preds_model":    preds_model,
        "preds_ensemble": preds_ensemble,
        "trues":          y_true_test,
    })

    # Reportar por gas/horizonte: baseline elegido, alpha, RMSE en test
    print(f"  Fold {held_out}: RMSE en TEST por gas/horizonte")
    for gi, gas in enumerate(POLLUTANTS):
        cfg = best_per_gas[gas]
        for hi, h in enumerate(HORIZONS):
            r_b = _rmse(preds_baseline[:, hi, gi], y_true_test[:, hi, gi])
            r_m = _rmse(preds_model[:, hi, gi],    y_true_test[:, hi, gi])
            r_e = _rmse(preds_ensemble[:, hi, gi], y_true_test[:, hi, gi])
            n   = int((np.isfinite(preds_ensemble[:, hi, gi]) &
                       np.isfinite(y_true_test[:, hi, gi])).sum())
            print(f"    {gas:3s} T+{h}: base[{cfg['baseline']:9s}]={r_b:6.2f}  "
                  f"+resid={r_m:6.2f}  +{cfg['alpha']:.2f}*resid={r_e:6.2f}  n={n}")

print(f"\nLOO-CV completo: {len(resultados_loo)} folds")


Conv3DSit3 ligero: 7,921 parametros (predice RESIDUOS)

LOO-CV con 9 estaciones: ['Base_Aerea', 'Canaveralejo', 'Compartir', 'ERA_-_Obrero', 'Ermita', 'Flora', 'Pance', 'Transitoria_-_Navarro', 'Univalle']

  Fold LOO: dejando fuera Base_Aerea
  Train: 1361 seqs, Test: 217 seqs
    Ep 10  train=1.0014  val=1.1132
    Ep 20  train=0.9890  val=1.1131
    Ep 30  train=0.9824  val=1.1131
    Ep 40  train=0.9924  val=1.1131
  Fold Base_Aerea: RMSE en TEST por gas/horizonte
    NO2 T+1: base[ewma     ]=   nan  +resid=   nan  +0.00*resid=   nan  n=0
    NO2 T+3: base[ewma     ]=   nan  +resid=   nan  +0.00*resid=   nan  n=0
    NO2 T+7: base[ewma     ]=   nan  +resid=   nan  +0.00*resid=   nan  n=0
    SO2 T+1: base[persist  ]=  2.52  +resid=  2.52  +0.00*resid=  2.52  n=162
    SO2 T+3: base[persist  ]=  2.56  +resid=  2.56  +0.00*resid=  2.56  n=157
    SO2 T+7: base[persist  ]=  2.77  +resid=  2.77  +0.00*resid=  2.77  n=151
    O3  T+1: base[ewma     ]=  4.90  +resid=  4.90  +0.30*resid= 

In [19]:
# @title 8c. Metricas agregadas LOO-CV (mejor metodo por gas)
# Pool todos los folds en un solo vector por gas/horizonte y se selecciona,
# para cada gas, el METODO con menor RMSE T+1. Esa misma eleccion se reporta
# para T+3 y T+7 (degradacion). Tambien se muestran los 4 metodos lado a lado.

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np
import pandas as pd

KPI_UGM3 = {"NO2": 8.0, "SO2": 6.0, "O3": 12.0}
METODOS = ["preds_baseline", "preds_model", "preds_ensemble"]
METODO_LABEL = {
    "preds_baseline": "baseline",
    "preds_model":    "+resid(1.0)",
    "preds_ensemble": "+resid(a*)",
}


def _pool(metodo, gi, hi):
    """Concatena predicciones y verdad para un (gas, horizonte) sobre todos los folds."""
    preds, trues = [], []
    for fold in resultados_loo:
        p = fold[metodo][:, hi, gi]
        t = fold["trues"][:, hi, gi]
        m = np.isfinite(p) & np.isfinite(t)
        if m.any():
            preds.append(p[m])
            trues.append(t[m])
    if not preds:
        return np.array([]), np.array([])
    return np.concatenate(preds), np.concatenate(trues)


def _metricas(p, t):
    if len(p) < 2:
        return float("nan"), float("nan"), float("nan")
    rmse = float(np.sqrt(mean_squared_error(t, p)))
    mae  = float(mean_absolute_error(t, p))
    try:
        r2 = float(r2_score(t, p))
    except Exception:
        r2 = float("nan")
    return rmse, mae, r2


print("=" * 78)
print("  LOO-CV DAGMA - RMSE por gas y horizonte (3 metodos)")
print("=" * 78)
print(f"  {'Gas':4s} {'h':3s} {'n':>4s} "
      f"{'baseline':>10s} {'+resid':>10s} {'+a*resid':>10s} "
      f"{'mejor':>10s} {'metodo':>14s}")
print("  " + "-" * 76)

filas = []
mejor_por_gas = {}  # gas -> metodo ganador para T+1

for gi, gas in enumerate(POLLUTANTS):
    for hi, h in enumerate(HORIZONS):
        rmses = {}
        maes = {}
        r2s = {}
        ns = {}
        for met in METODOS:
            p, t = _pool(met, gi, hi)
            rmse, mae, r2 = _metricas(p, t)
            rmses[met] = rmse
            maes[met]  = mae
            r2s[met]   = r2
            ns[met]    = len(p)
        if all(not np.isfinite(v) for v in rmses.values()):
            continue
        met_best = min((m for m in METODOS if np.isfinite(rmses[m])),
                       key=lambda m: rmses[m])
        if h == 1:
            mejor_por_gas[gas] = met_best
        print(f"  {gas:4s} T+{h:<1d} {ns[met_best]:>4d} "
              f"{rmses['preds_baseline']:>10.2f} {rmses['preds_model']:>10.2f} "
              f"{rmses['preds_ensemble']:>10.2f} "
              f"{rmses[met_best]:>10.2f} {METODO_LABEL[met_best]:>14s}")
        filas.append({
            "gas": gas, "horizonte": f"T+{h}", "n": ns[met_best],
            "rmse_baseline": rmses["preds_baseline"],
            "rmse_model":    rmses["preds_model"],
            "rmse_ensemble": rmses["preds_ensemble"],
            "rmse_best":     rmses[met_best],
            "mae_best":      maes[met_best],
            "r2_best":       r2s[met_best],
            "metodo_best":   METODO_LABEL[met_best],
        })

resumen = pd.DataFrame(filas)

# ============================================================
# Para cada gas, fijamos el metodo ganador en T+1 y reportamos
# RMSE/MAE/R2 en T+1, T+3, T+7 con ese mismo metodo (sin "cherry-pick"
# por horizonte). Esto es lo que se compara contra el KPI.
# ============================================================
print("\n" + "=" * 78)
print("  Mejor metodo por gas (elegido en T+1, aplicado a los 3 horizontes)")
print("=" * 78)
print(f"  {'Gas':4s} {'metodo':>14s} "
      f"{'RMSE T+1':>10s} {'RMSE T+3':>10s} {'RMSE T+7':>10s} "
      f"{'MAE T+1':>10s} {'R2 T+1':>10s} {'KPI':>6s} {'cumple':>7s}")
print("  " + "-" * 84)

filas_fix = []
for gi, gas in enumerate(POLLUTANTS):
    if gas not in mejor_por_gas:
        continue
    met = mejor_por_gas[gas]
    rmses_h = {}
    for hi, h in enumerate(HORIZONS):
        p, t = _pool(met, gi, hi)
        rmse, mae, r2 = _metricas(p, t)
        rmses_h[h] = (rmse, mae, r2, len(p))
    rmse_t1, mae_t1, r2_t1, n_t1 = rmses_h[1]
    rmse_t3, _, _, _ = rmses_h[3]
    rmse_t7, _, _, _ = rmses_h[7]
    cumple = "SI" if (np.isfinite(rmse_t1) and rmse_t1 <= KPI_UGM3[gas]) else "NO"
    print(f"  {gas:4s} {METODO_LABEL[met]:>14s} "
          f"{rmse_t1:>10.2f} {rmse_t3:>10.2f} {rmse_t7:>10.2f} "
          f"{mae_t1:>10.2f} {r2_t1:>10.4f} {KPI_UGM3[gas]:>6.1f} {cumple:>7s}")
    for h in HORIZONS:
        r, m, r2, n = rmses_h[h]
        filas_fix.append({
            "gas": gas, "metodo": METODO_LABEL[met], "horizonte": f"T+{h}",
            "rmse": r, "mae": m, "r2": r2, "n": n,
        })

resumen_fix = pd.DataFrame(filas_fix)

# ============================================================
# Tabla por estacion (para reporte LOO-CV completo)
# ============================================================
print("\n" + "=" * 78)
print("  RMSE T+1 por estacion (metodo ganador por gas)")
print("=" * 78)
filas_est = []
for fold in resultados_loo:
    est = fold["estacion"]
    fila = {"estacion": est, "n_test": fold["n_test"]}
    for gi, gas in enumerate(POLLUTANTS):
        if gas not in mejor_por_gas:
            fila[f"RMSE_{gas}_T1"] = float("nan")
            continue
        met = mejor_por_gas[gas]
        p = fold[met][:, 0, gi]
        t = fold["trues"][:, 0, gi]
        m = np.isfinite(p) & np.isfinite(t)
        fila[f"RMSE_{gas}_T1"] = (float(np.sqrt(np.mean((p[m] - t[m]) ** 2)))
                                  if m.sum() >= 2 else float("nan"))
    filas_est.append(fila)
df_estaciones = pd.DataFrame(filas_est)
print(df_estaciones.to_string(index=False, float_format=lambda x: f"{x:6.2f}"))

# ============================================================
# Degradacion T+1 -> T+7 (del metodo ganador)
# ============================================================
print("\nDegradacion T+1 -> T+7 (RMSE, KPI: < 60% aumento):")
for gas in POLLUTANTS:
    sub = resumen_fix[resumen_fix.gas == gas]
    if sub.empty:
        continue
    r1 = float(sub[sub.horizonte == "T+1"].rmse.values[0])
    r7 = float(sub[sub.horizonte == "T+7"].rmse.values[0])
    if r1 > 0 and np.isfinite(r7):
        deg = (r7 / r1 - 1.0) * 100.0
        flag = "OK" if deg < 60 else "ALTO"
        print(f"  {gas}: {deg:+.1f}%  [{flag}]")
    else:
        print(f"  {gas}: N/A")

# ============================================================
# Resumen final KPI
# ============================================================
print("\n" + "=" * 78)
print("  COMPARACION KPI - Situacion 3 (LOO-CV DAGMA)")
print("=" * 78)
cumplidos = 0
total = 0
for gas, kpi in KPI_UGM3.items():
    sub = resumen_fix[(resumen_fix.gas == gas) & (resumen_fix.horizonte == "T+1")]
    if sub.empty:
        print(f"  RMSE {gas} T+1: sin datos")
        continue
    r = float(sub.rmse.values[0])
    cumple = r <= kpi
    cumplidos += int(cumple)
    total += 1
    print(f"  RMSE {gas} T+1: {r:6.2f} ug/m3   KPI <= {kpi:5.1f}   -> {'SI' if cumple else 'NO'}")
r2_prom = float(resumen_fix[resumen_fix.horizonte == "T+1"].r2.mean())
print(f"  R2 promedio T+1: {r2_prom:.4f}   KPI >= 0.55   -> {'SI' if r2_prom >= 0.55 else 'NO (con tan pocos datos por estacion es normal)'}")
print(f"\n  KPIs RMSE cumplidos: {cumplidos}/{total}")

# Guardar resultados
try:
    OUT_DIR = Path("/kaggle/working/loocv_dagma")
except Exception:
    OUT_DIR = Path("/content/loocv_dagma")
OUT_DIR.mkdir(parents=True, exist_ok=True)
resumen.to_csv(OUT_DIR / "metricas_loocv_dagma_completo.csv", index=False)
resumen_fix.to_csv(OUT_DIR / "metricas_loocv_dagma_kpi.csv", index=False)
df_estaciones.to_csv(OUT_DIR / "rmse_por_estacion.csv", index=False)
print(f"\nResultados guardados en {OUT_DIR}")


  LOO-CV DAGMA - RMSE por gas y horizonte (3 metodos)
  Gas  h      n   baseline     +resid   +a*resid      mejor         metodo
  ----------------------------------------------------------------------------
  NO2  T+1  257       4.96       4.96       4.96       4.96       baseline
  NO2  T+3  251       5.43       5.43       5.43       5.43       baseline
  NO2  T+7  254       5.75       5.75       5.75       5.75       baseline
  SO2  T+1  634       3.52       3.52       3.52       3.52    +resid(1.0)
  SO2  T+3  615       3.67       3.67       3.67       3.67    +resid(1.0)
  SO2  T+7  602       4.11       4.11       4.11       4.11    +resid(1.0)
  O3   T+1 1104       9.31       9.31       9.31       9.31     +resid(a*)
  O3   T+3 1073      10.77      10.78      10.78      10.77       baseline
  O3   T+7 1056      10.25      10.27      10.26      10.25       baseline

  Mejor metodo por gas (elegido en T+1, aplicado a los 3 horizontes)
  Gas          metodo   RMSE T+1   RMSE T+3   R

## 9. Diagnostico, limitaciones y plan de mejoras

### Por que el LOO-CV daba peor que el 70/15/15

El split 70/15/15 aleatorio del notebook original generaba **leakage** porque las
1403 secuencias se construyen con ventana deslizante de 8 timesteps sobre tiles
que comparten ubicacion: secuencias contiguas comparten 7/8 de input y predicen
targets a 1-7 dias casi identicos por autocorrelacion. El modelo memorizaba el
tile en vez de generalizar (RMSE NO2 = 0.08 ug/m³ no es real; el ruido propio
de DAGMA ya es ~3 ug/m³). Por eso aqui usamos:

1. **Split honesto** por bloques contiguos / tile_id (celda 9).
2. **LOO-CV con DAGMA** como validacion principal (celdas 24-27).

### Limitaciones que conviene declarar en el informe

- **Mismatch S5P vs DAGMA**: el target de entrenamiento (Sentinel-5P, columna
  troposferica en mol/m², resolucion ~7 km) y la validacion (DAGMA, superficie
  en ug/m³, puntual y horario→mediana diaria) son cantidades correlacionadas
  pero distintas. Parte del RMSE LOO-CV es ese mismatch fisico, no error del
  modelo. Una funcion de transferencia (e.g. regresion lineal por estacion
  S5P→DAGMA) es trabajo futuro.
- **NO2 con una sola estacion DAGMA (Univalle)**: en LOO el train se queda sin
  NO2; la metrica NO2 del fold "Univalle" es de baja confiabilidad estructural.
- **9 estaciones DAGMA** son pocas para un R² LOO-CV positivo robusto; por eso
  el KPI principal alcanzable es **RMSE por contaminante**, no R².

### Mejoras priorizadas (orden de prioridad de implementacion)

| # | Cambio | Lo que se espera |
|---|--------|------------------|
| 1 | Split por bloques / tile_id (ya aplicado) | Test honesto; RMSE en test sube pero ahora refleja generalizacion |
| 2 | Persistencia + correccion ligera del Conv3D (celdas 25-27) | Cumple KPI RMSE NO2/SO2/O3 T+1 en LOO-CV gracias a la autocorrelacion |
| 3 | Drop de los 512 canales CLIP en una variante "physical-only" | Conserva solo los 19 canales fisicos (ERA5, MODIS, NDVI/BSI/NDBI, temporal, lat/lon). El Conv3D aprende sobre senal fisica y el LOO-CV es mas justo |
| 4 | Reemplazar 70/15/15 normal por **GroupKFold por tile_id** y reportar media ± std de los folds | Estimador robusto de la generalizacion espacial |
| 5 | Cambiar Conv3D por **GRU/Transformer** sobre los 19 canales fisicos pooleados 5x5→1 | Reduce parametros 100x, evita overfitting, mejor LOO |

### Que NO va a funcionar (no perder tiempo)

- Aumentar capacidad del Conv3D (LSTM hidden=256, mas capas) sin arreglar el
  split: amplifica memorizacion.
- Data augmentation espacial (flip/rotate) sobre tiles 5x5 de Cali:
  rompe el contexto geografico (lat/lon son canales) y empeora LOO.
- Cambiar de InfoNCE/CLIP a otro encoder visual: el embedding visual no es la
  senal limitante; las **covariables fisicas** son las que hay que potenciar.

### Plan minimo viable para el informe (orden de ejecucion)

1. Re-ejecutar el notebook con el split honesto (celdas 7-15) → reportar el
   nuevo "test" como baseline de referencia (no como KPI).
2. Ejecutar **celdas 24-27** (LOO-CV con persistencia + correccion). Reportar
   RMSE T+1 por contaminante vs KPI (tabla final de la celda 8c).
3. Adjuntar `training_curves.png` (celda 18) y los CSV de `loocv_dagma/` como
   evidencia.
4. En el informe, declarar las 3 limitaciones de la seccion anterior.